# 01 — Data Collection

This notebook collects the raw data for the arboviral outbreak prediction project.
Each source has its own section: a short explanation, the download call, and a
visible sample of what came back.

**Design rules**
- Each run saves to a dated folder `data/raw/<date>/`, and later notebooks read
  from there. We never overwrite a previous run.
- Two files are **manual assets** and deliberately live outside that dated
  folder, in `data/lookup/`: `municipios_sp_estacoes_inmet.xlsx` (one
  latitude/longitude per municipality, and the 6→7-digit code lookup) and
  `idhm_2010_sp.csv`. Nothing here downloads them — they are versioned once and
  read from a stable path, because a hand-placed file in a dated folder would
  have to be copied in again on every run.
- Raw data is saved **exactly as the source delivers it** — no filtering, no
  renaming of columns. Portuguese column names (`UF_LPI`, `CASCONF`, ...) are kept
  as-is because they *are* the data. Cleaning happens later, in `02_master`.
- If a download fails, the cell stops with a clear error instead of continuing
  with half a file. The single exception is the IDH-M half of Source 9, which
  cannot be collected automatically at all and reports its absence instead.

In [13]:
from pathlib import Path
from datetime import date
import requests
import pandas as pd

# --- Project paths ---
# The notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent
RUN_DATE = date.today().isoformat()                 # e.g. "2026-09-09"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / RUN_DATE

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run date     : {RUN_DATE}")
print(f"Saving raw to : {RAW_DIR.relative_to(PROJECT_ROOT)}")

In [ ]:
# --- Prerequisites ---
# Checked up front so nothing surprises us halfway through a long download.
#   Python packages — installed automatically when missing.
#   System tools    — only reported, because installing them needs apt and sudo,
#                     and every simple source works fine without them.
import importlib
import shutil
import subprocess
import sys

def check_tool(name, purpose):
    path = shutil.which(name)
    status = "OK" if path else "MISSING"
    print(f"  [{status:7}] {name:15} — {purpose}")
    return path is not None

print("System tools:")
has_7z     = check_tool("7z",           "unpack .rar/.zip (SINISA source)")
has_chrome = check_tool("chromedriver", "Selenium browser (yellow-fever vaccination source)")
if not has_7z:
    print("    → sudo apt install p7zip-full p7zip-rar")
if not has_chrome:
    # Not actually a blocker: Selenium 4.6+ ships Selenium Manager, which fetches
    # a chromedriver matching the installed Chrome on first use. Chrome itself,
    # however, does have to be there.
    print("    → not on PATH; Selenium will fetch a matching one by itself")
    print(f"    → Chrome itself: {shutil.which('google-chrome') or shutil.which('chromium') or 'MISSING — install it'}")

print("\nPython packages:")
# (import name, pip name, what needs it) — the two names differ for calamine.
for module, package, purpose in [
    ("requests",        "requests",        "every HTTP download"),
    ("pandas",          "pandas",          "reading and previewing tables"),
    ("pyarrow",         "pyarrow",         "parquet output in the later notebooks"),
    ("python_calamine", "python-calamine", "old .xls files (territorial areas)"),
    # DATASUS ships .dbc, its own compressed DBF. pyreaddbc expands it and
    # dbfread reads the result. pyreaddbc builds a C extension, so it needs a
    # compiler (build-essential) available.
    ("pyreaddbc",       "pyreaddbc",       ".dbc files from DATASUS (SIH-SUS, SINAN)"),
    ("dbfread",         "dbfread",         "the .dbf that comes out of a .dbc"),
    ("selenium",        "selenium",        "the Qlik panel (yellow-fever vaccination)"),
]:
    try:
        __import__(module)
        print(f"  [OK     ] {package:16} — {purpose}")
    except ImportError:
        print(f"  [INSTALL] {package:16} — {purpose}; installing now...", flush=True)
        # sys.executable, never a bare `pip`: that is the interpreter this kernel
        # is actually running, which is not always the one first on PATH.
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)          # a failed install stops the cell here
        # A package installed after the interpreter started is invisible until
        # the import caches are refreshed.
        importlib.invalidate_caches()
        __import__(module)                  # prove it imports in THIS session
        print(f"  [OK     ] {package:16} — installed")

## Source 1 — Yellow fever (Ministry of Health, open data)

A single national CSV with every confirmed human case since 1994. Unlike the
urban arboviruses, yellow fever is not in SINAN: because it's sylvatic and rare,
the Ministry of Health publishes it through a separate open-data portal.

Two things that matter about this source:
- The file is **latin1**-encoded and **semicolon**-separated — we must say so when
  reading it, or the accents and columns come out wrong.
- The municipality in this file is the **probable place of infection (LPI)**, not
  the patient's residence. That's correct for a sylvatic disease (people are
  usually infected away from home), but it differs from SINAN, which uses
  residence. We'll check how many rows actually have LPI filled in `02_master`.

The filename ends in the most recent year (e.g. `...1994-2026.csv`) and changes
with each update, so we first probe which version exists today.

In [15]:
# --- Source 1: Yellow fever ---
YELLOW_FEVER_BASE = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/Febre+Amarela"

source_dir = RAW_DIR / "yellow_fever"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: find which version of the file exists today.
# The name ends in the latest year, which changes, so we try current year backwards.
current_year = date.today().year
file_url = None
for year in range(current_year, current_year - 3, -1):     # try 2026, 2025, 2024
    candidate = f"{YELLOW_FEVER_BASE}/fa_casoshumanos_1994-{year}.csv"
    if requests.head(candidate, timeout=30).status_code == 200:
        file_url = candidate
        print(f"Found version: series 1994–{year}")
        break

if file_url is None:
    raise RuntimeError("No yellow-fever version found in the last 3 years.")

# Step 2: download the CSV (the full series since 1994).
destination = source_dir / file_url.split("/")[-1]
response = requests.get(file_url, timeout=120)
response.raise_for_status()          # stop here with a clear error if the download fails
destination.write_bytes(response.content)
print(f"Downloaded: {destination.name} ({destination.stat().st_size // 1024} KB)")

# Step 3: open it and look at a sample, to confirm it arrived correctly.
yellow_fever = pd.read_csv(destination, sep=";", encoding="latin1", low_memory=False)
print(f"\nRows: {len(yellow_fever):,}")
print(f"Columns: {list(yellow_fever.columns)}")
yellow_fever.head(3)

Found version: series 1994–2026
Downloaded: fa_casoshumanos_1994-2026.csv (221 KB)

Rows: 2,912
Columns: ['ID', 'MACRORREG_LPI', 'COD_UF_LPI', 'UF_LPI', 'COD_MUN_LPI', 'MUN_LPI', 'SEXO', 'IDADE', 'DT_IS', 'SE_IS', 'MES_IS', 'ANO_IS', 'MONITORAMENTO_IS', 'OBITO', 'DT_OBITO', 'Unnamed: 15']


,ID,MACRORREG_LPI,COD_UF_LPI,UF_LPI,COD_MUN_LPI,MUN_LPI,SEXO,IDADE,DT_IS,SE_IS,MES_IS,ANO_IS,MONITORAMENTO_IS,OBITO,DT_OBITO,Unnamed: 15
0,1.0,N,14.0,RR,140005.0,ALTO ALEGRE,M,NaN,29/11/1994,48.0,11.0,1994.0,1994/1995,SIM,01/12/1994,NaN
1,2.0,N,14.0,RR,140045.0,PACARAIMA,M,19,19/02/1995,8.0,2.0,1995.0,1994/1995,NÃO,NaN,NaN
2,3.0,NE,21.0,MA,210060.0,AMARANTE DO MARANHÃO,M,32,01/04/1995,13.0,4.0,1995.0,1994/1995,IGN,NaN,NaN


## Source 2 — Commuting between municipalities (IBGE, two censuses)

How many people leave their own municipality every day to work or study
somewhere else. This matters here because arboviruses travel with people: a
municipality that sends thousands of commuters into its neighbour every morning
is epidemiologically much closer to it than the map suggests.

IBGE publishes this in two completely different shapes, one per census, and we
take both:

- **Census 2010 — sample microdata** (~200 MB in three `.zip` files). One row per
  person surveyed, and crucially the **destination** municipality. That is what
  lets us count both who LEAVES and who ARRIVES in each municipality.
- **Census 2022 — SIDRA table 10329** (a small JSON from IBGE's API). Only the
  aggregate "went to another municipality", by municipality of residence.

Things that matter about this source:

- **The two censuses are not symmetric.** The 2022 table carries no destination
  at all, and the 2022 microdata were still not public as of August 2026, so
  *arrivals* simply do not exist from 2022 on. That is a limit of the source, not
  something to patch — we record the gap instead of inventing a number for it.
- **The files are big and immutable.** A census does not get revised, so a file
  already on disk is the right file: the cell skips it and re-running is cheap.
- **A download that dies mid-stream raises no error.** The reader just stops and
  leaves a silently truncated `.zip` behind — the worst possible failure mode for
  data. We write under a `.partial` name, compare the bytes we actually got
  against the size the server promised, and only then rename to the real file.
- **SIDRA sometimes answers gzip-compressed without being asked.** `requests`
  unpacks that transparently; hand-rolled `urllib` code chokes on the very first
  byte (`0x8b`, the gzip signature). Worth knowing if you ever swap the library.
- **This is a census source, so it is static.** Nothing new arrives between
  censuses, so it never holds back how fresh the rest of the database can be.

The 2010 microdata are fixed-width text files that need the layout dictionary in
`Documentacao.zip` to be read at all — which is why that archive comes along.
Here we only download and list what is inside; parsing happens in `02_master`.

In [16]:
# --- Source 2: Commuting between municipalities ---
import json
import zipfile

CENSUS_2010_BASE = (
    "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2010/"
    "Resultados_Gerais_da_Amostra/Microdados"
)
CENSUS_2010_FILES = ("SP1.zip", "SP2_RM.zip", "Documentacao.zip")

# SIDRA aggregate 10329, variable 13373, for every municipality (N6) inside São
# Paulo state (N3 code 35). In the classification filter, 469[12188] is "Outro
# município" — the people who commute OUT. The other three (2[6794], 386[9680],
# 2087[79177]) pin the remaining breakdowns to their "Total" category, so each
# municipality comes back as a single number. The codes are carried over
# verbatim; what each classification means is listed at
# https://servicodados.ibge.gov.br/api/v3/agregados/10329/metadados
CENSUS_2022_URL = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/10329/"
    "periodos/2022/variaveis/13373"
    "?localidades=N6[N3[35]]"
    "&classificacao=469[12188]|2[6794]|386[9680]|2087[79177]"
)

source_dir = RAW_DIR / "commuting"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: the 2010 microdata — about 200 MB in total, so this is the slow part.
for filename in CENSUS_2010_FILES:
    destination = source_dir / filename

    # A census never changes, so a file we already have is the right file.
    if destination.exists():
        print(f"Already here : {filename} ({destination.stat().st_size / 1e6:.0f} MB)")
        continue

    response = requests.get(f"{CENSUS_2010_BASE}/{filename}", stream=True, timeout=1800)
    response.raise_for_status()          # a 404 or 5xx stops the cell right here
    expected = int(response.headers.get("Content-Length") or 0)

    # Write under a temporary name first: a half-written file must never sit
    # there under the real name, looking perfectly valid to the next notebook.
    partial = destination.with_name(filename + ".partial")
    written = 0
    with open(partial, "wb") as out:
        for chunk in response.iter_content(chunk_size=1 << 20):     # 1 MB at a time
            out.write(chunk)
            written += len(chunk)

    # A connection cut mid-stream raises nothing — the loop just ends early.
    # Comparing against the size the server promised is what catches it.
    if expected and written != expected:
        partial.unlink()
        raise RuntimeError(
            f"{filename}: truncated download, got {written:,} of {expected:,} bytes")

    partial.rename(destination)
    print(f"Downloaded   : {filename} ({written / 1e6:.0f} MB)")

# Step 2: the 2022 aggregate — one small JSON, saved exactly as the API sends it.
destination_2022 = source_dir / "sidra_10329_saidas_2022.json"
response = requests.get(CENSUS_2022_URL, timeout=300)
response.raise_for_status()
commuting_2022_json = response.json()    # requests undoes the gzip case for us
destination_2022.write_text(
    json.dumps(commuting_2022_json, ensure_ascii=False), encoding="utf-8")
print(f"Downloaded   : {destination_2022.name} "
      f"({destination_2022.stat().st_size // 1024} KB)")

# Step 3: look at what arrived.
# For 2010 we can only list the archives: the microdata inside are fixed-width
# text that needs the layout dictionary, and reading it is 02_master's job.
print("\n2010 microdata — contents of each archive:")
for filename in CENSUS_2010_FILES:
    with zipfile.ZipFile(source_dir / filename) as archive:
        names = archive.namelist()
    print(f"  {filename}: {len(names)} file(s), e.g. {names[:3]}")

# For 2022 we flatten the JSON into a table just to eyeball it. The file on disk
# stays untouched, and the keys below (localidade, serie) are SIDRA's own.
series = commuting_2022_json[0]["resultados"][0]["series"]
commuting_2022 = pd.DataFrame([
    {
        "localidade_id": item["localidade"]["id"],
        "localidade_nome": item["localidade"]["nome"],
        "serie_2022": item["serie"]["2022"],
    }
    for item in series
])
print(f"\n2022 SIDRA — rows: {len(commuting_2022):,}  (SP has 645 municipalities)")
print(f"Columns: {list(commuting_2022.columns)}")
commuting_2022.head(3)

Already here : SP1.zip (137 MB)
Already here : SP2_RM.zip (67 MB)
Already here : Documentacao.zip (11 MB)
Downloaded   : sidra_10329_saidas_2022.json (85 KB)

2010 microdata — contents of each archive:
  SP1.zip: 5 file(s), e.g. ['SP1/', 'SP1/Amostra_Domicilios_35_outras.txt', 'SP1/Amostra_Emigracao_35_outras.txt']
  SP2_RM.zip: 5 file(s), e.g. ['SP2-RM/', 'SP2-RM/Amostra_Domicilios_35_RMSP.txt', 'SP2-RM/Amostra_Emigracao_35_RMSP.txt']
  Documentacao.zip: 53 file(s), e.g. ['Documentaç╞o/Anexos Auxiliares/', 'Documentaç╞o/Anexos Auxiliares/Atividade CNAE_DOM 2.0 2010.ods', 'Documentaç╞o/Anexos Auxiliares/Atividade CNAE_DOM 2.0 2010.xls']

2022 SIDRA — rows: 645  (SP has 645 municipalities)
Columns: ['localidade_id', 'localidade_nome', 'serie_2022']


,localidade_id,localidade_nome,serie_2022
0,3500105,Adamantina - SP,939
1,3500204,Adolfo - SP,413
2,3500303,Aguaí - SP,783


## Source 3 — Housing: favelas and subnormal settlements (IBGE, SIDRA API)

How many favelas each municipality has, and how many people live in them. This
is here because precarious, dense housing with irregular water supply is exactly
where *Aedes aegypti* breeds best: water stored in drums and tanks because the
tap is unreliable is the mosquito's preferred nursery.

IBGE publishes it through SIDRA, its aggregate-tables API. Four queries, because
the two censuses are organised differently:

| Census | Table | Variable | What it is |
|---|---|---|---|
| 2010 | 3379 | 744 | number of *aglomerados subnormais* |
| 2010 | 3379 | 743 | population living in them |
| 2022 | 9883 | 9910 | number of *favelas e comunidades urbanas* |
| 2022 | 9900 | 9612 | population living in them |

Things that matter about this source:

- **The concept itself changed between the censuses.** IBGE renamed "aglomerado
  subnormal" (2010) to "favela e comunidade urbana" (2022) and adjusted how the
  areas are delimited. In São Paulo the municipalities with any record go from
  **60 to 92**. Part of that jump is real growth and part is the new definition —
  and there is no way to separate the two. We treat it as one variable over time
  and carry the caveat with it; the cell prints both counts so the break is
  visible rather than buried.
- **A municipality missing from the table means NO RECORD, not zero.** Filling it
  with zero would assert "there is no favela here", when the census merely did
  not identify one. It stays null and the zero stays implicit. This is the kind
  of thing that quietly ruins a model, so it is worth re-checking in `02_master`.
- **SIDRA never sends an empty cell — it sends a code saying why the cell is
  empty**: `-` (the phenomenon does not exist), `..` (does not apply), `...`
  (not available), `X` (withheld for confidentiality). Every one of them becomes
  null, never zero. The raw JSON on disk keeps them verbatim; only the preview
  table below converts.
- **You cannot refresh this source by bumping a year.** Each census lives in its
  own table with its own variable ids, so the pairs are pinned per census and a
  future 2030 census will need new ones. That is why the years are hard-coded
  here and nowhere else.

`N6[N3[35]]` in the URL means "municipalities (N6) inside state 35 (N3)" — all
645 paulista municipalities in a single request, instead of pulling the country's
5,570 and filtering afterwards.

**What lands on disk:** the four raw JSON answers, kept as provenance, plus
`sidra_housing.csv` — the numbers themselves, one row per municipality per
census, with `num_favelas` and `pop_favelas` side by side. The raw JSON is the
record of what IBGE actually sent; the CSV is the part you can read. A
municipality with no record simply has no row, which is the null-not-zero rule
above expressed as a table.

In [17]:
# --- Source 3: Housing (favelas / aglomerados subnormais) ---
import json

SIDRA_BASE = "https://servicodados.ibge.gov.br/api/v3/agregados"

# (table, variable, census year, what it measures)
HOUSING_QUERIES = [
    (3379,  744, 2010, "number of aglomerados subnormais"),
    (3379,  743, 2010, "population in aglomerados subnormais"),
    (9883, 9910, 2022, "number of favelas e comunidades urbanas"),
    (9900, 9612, 2022, "population in favelas e comunidades urbanas"),
]

# SIDRA never leaves a cell blank: it sends a code explaining WHY it is blank.
# All of them mean "no number", none of them mean zero.
NON_NUMERIC = {"-", "..", "...", "X", "x", ""}

source_dir = RAW_DIR / "housing"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: one request per (table, variable), each saved exactly as SIDRA sends it.
preview_rows = []
for table, variable, year, meaning in HOUSING_QUERIES:
    url = (f"{SIDRA_BASE}/{table}/periodos/{year}/variaveis/{variable}"
           f"?localidades=N6[N3[35]]")
    response = requests.get(url, timeout=300)
    response.raise_for_status()
    payload = response.json()

    # An empty answer here means the query is wrong or the table moved — it does
    # NOT mean "no favelas". Stop, rather than write an empty file and move on.
    if not payload or not payload[0].get("resultados"):
        raise RuntimeError(
            f"SIDRA table {table}, variable {variable} ({year}) came back empty")

    destination = source_dir / f"sidra_{table}_{variable}_{year}.json"
    # indent=2 so the file is actually readable when you open it: SIDRA sends
    # everything on one line and the numbers get lost in the envelope.
    destination.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    # Step 2: flatten this answer for inspection only — the file on disk is
    # untouched. The keys (localidade, serie) are SIDRA's own.
    rows_before = len(preview_rows)
    for block in payload[0]["resultados"]:
        for item in block.get("series", []):
            raw_value = item["serie"].get(str(year))
            preview_rows.append({
                "tabela": table,
                "variavel": variable,
                "ano": year,
                "localidade_id": item["localidade"]["id"],
                "localidade_nome": item["localidade"]["nome"],
                "valor_bruto": raw_value,       # kept as SIDRA wrote it
                "valor": None if str(raw_value).strip() in NON_NUMERIC else raw_value,
            })

    returned = preview_rows[rows_before:]
    with_record = sum(1 for row in returned if row["valor"] is not None)
    print(f"table {table:>4} var {variable:>4} ({year}): "
          f"{len(returned):>3} municipalities returned, "
          f"{with_record:>3} with a record — {meaning}")

# Step 3: the numbers themselves, written out as a table.
# The JSON above is the provenance record, but it is mostly API envelope — the
# values sit inside "serie", buried among localidade/nivel boilerplate. So the
# actual data is saved next to it, in the shape the rest of the project uses.
housing = pd.DataFrame(preview_rows)
housing["valor"] = pd.to_numeric(housing["valor"], errors="coerce")

# Each variable id says what its rows measure. 744 (2010) and 9910 (2022) count
# settlements; 743 (2010) and 9612 (2022) count the people living in them. The
# census is already carried in "ano", so both stack into the same two columns.
MEASURES = {744: "num_favelas", 9910: "num_favelas",
            743: "pop_favelas", 9612: "pop_favelas"}
housing["medida"] = housing["variavel"].map(MEASURES)

# Dropping the null values here is what "absent means no record, not zero"
# looks like in practice: a municipality without a record gets no row at all,
# instead of a row full of zeros that would claim it has no favelas.
housing_table = (housing.dropna(subset=["valor"])
                 .pivot(index=["localidade_id", "localidade_nome", "ano"],
                        columns="medida", values="valor")   # raises on duplicates
                 .reset_index()
                 .rename_axis(columns=None))
for column in ("num_favelas", "pop_favelas"):
    if column not in housing_table.columns:
        housing_table[column] = pd.NA
    # These are counts of settlements and of people — integers, not decimals.
    # Int64 (capital I) is the nullable one, so a missing value stays missing.
    housing_table[column] = housing_table[column].astype("Int64")
housing_table = (housing_table[["localidade_id", "localidade_nome", "ano",
                                "num_favelas", "pop_favelas"]]
                 .sort_values(["localidade_id", "ano"])
                 .reset_index(drop=True))

table_path = source_dir / "sidra_housing.csv"
housing_table.to_csv(table_path, index=False)

# Step 4: look at what arrived.
print(f"\nWrote {table_path.name}: {len(housing_table):,} rows, "
      f"columns {list(housing_table.columns)}")

# The concept change, made visible: how many SP municipalities have any favela
# record in each census. Expect roughly 60 in 2010 against 92 in 2022 — that gap
# is the definition change as much as it is growth.
print("\nMunicipalities with a favela record, by census:")
print(housing_table.groupby("ano")["localidade_id"].nunique().to_string())

housing_table.head(3)

table 3379 var  744 (2010):  60 municipalities returned,  60 with a record — number of aglomerados subnormais
table 3379 var  743 (2010):  60 municipalities returned,  60 with a record — population in aglomerados subnormais
table 9883 var 9910 (2022):  92 municipalities returned,  92 with a record — number of favelas e comunidades urbanas
table 9900 var 9612 (2022): 645 municipalities returned,  92 with a record — population in favelas e comunidades urbanas

Wrote sidra_housing.csv: 152 rows, columns ['localidade_id', 'localidade_nome', 'ano', 'num_favelas', 'pop_favelas']

Municipalities with a favela record, by census:
ano
2010    60
2022    92


,localidade_id,localidade_nome,ano,num_favelas,pop_favelas
0,3501608,Americana - SP,2022,1,481
1,3501905,Amparo - SP,2022,1,359
2,3503901,Arujá - SP,2022,4,1368


## Source 4 — Territorial areas, the base of population density (IBGE geoftp)

Careful with the name: **density itself is not downloaded anywhere**. What IBGE
publishes is the *area in km²* of every municipality in the country; density is
population ÷ area, and we compute it in `02_master` against the IBGE population
source. This section only fetches the denominator.

It matters because density is one of the strongest structural predictors we
have: *Aedes aegypti* has a flight range of barely a few hundred metres, so how
tightly people are packed decides how far one infected mosquito can carry an
outbreak.

The file lives on IBGE's geographic FTP, one folder per reference year:

```
.../areas_territoriais/2025/AR_BR_RG_UF_RGINT_RGI_MUN_2025.xls
```

Things that matter about this source:

- **The edition is discovered, never hard-coded.** We list the directory and take
  the newest year that works. This is not paranoia: the old standalone script had
  `--ano 2024` as its default and quietly kept serving 2024 areas long after
  IBGE had published 2025. A hard-coded year does not raise — it just goes stale.
- **The newest year's folder can exist while the file inside is not there yet.**
  So we check the *contents* of the folder, not that the folder exists, and fall
  back to the previous year when it is empty. A few editions also publish under a
  slightly different file name, so there is a second, looser match on `AR_BR*`.
- **The year is baked into three different names**: the file
  (`AR_BR_RG_UF_RGINT_RGI_MUN_2025.xls`), the sheet (`AR_BR_MUN_2025`) and the
  area column (`AR_MUN_2025`). Anything that hard-codes one of the three breaks
  the day the edition changes — and breaks *silently*, with a `KeyError` at best.
  We match all three by prefix.
- **It is the old binary `.xls`**, which pandas cannot open on its own. It needs
  `python-calamine`, which reads both `.xls` and `.xlsx`; `openpyxl` does **not**
  help here, as it only handles `.xlsx`. The prerequisites cell installs it, and
  this cell still checks for it up front so that a skipped prerequisites cell
  stops you before the download rather than after it.
- **geoftp sometimes answers `200 OK` with an HTML error page.** A minimum-size
  check catches that before an HTML page gets written out as a `.xls`.
- **`CD_MUN` arrives as a float** (`1100015.0`), with blank rows at the bottom of
  the sheet. Worth knowing before comparing municipality codes anywhere.
- **The file is national** — all 5,570 municipalities. It stays that way on disk;
  narrowing to São Paulo's 645 is `02_master`'s job, not the collector's.

Areas are effectively static — they change only with an emancipation, a merger
or a boundary revision — so this source never holds back how fresh the database
can be. What the discovery step buys us is simply not being stuck on an old
edition by forgetting.

In [18]:
# --- Source 4: Territorial areas (base of population density) ---
import importlib.util
import re

AREAS_BASE = ("https://geoftp.ibge.gov.br/organizacao_do_territorio/"
              "estrutura_territorial/areas_territoriais")
OLDEST_EDITION = 2020          # older editions are not worth falling back to
HREF = re.compile(r'href="([^"]+)"')

# Step 0: the file is the old binary .xls, which pandas cannot read by itself.
# Check now, before spending a network round-trip, and say exactly what to install.
if importlib.util.find_spec("python_calamine") is None:
    raise RuntimeError(
        "This cell needs an engine that reads the old .xls format.\n"
        "  pip install python-calamine\n"
        "(openpyxl will NOT do — it only reads .xlsx.)")

# Step 1: list the editions. One folder per reference year.
response = requests.get(f"{AREAS_BASE}/", timeout=120)
response.raise_for_status()
editions = sorted(
    {int(href.strip("/")) for href in HREF.findall(response.text)
     if href.strip("/").isdigit() and int(href.strip("/")) >= OLDEST_EDITION},
    reverse=True,
)
if not editions:
    raise RuntimeError(f"No edition folder from {OLDEST_EDITION} on at {AREAS_BASE}/")
print(f"Editions published: {editions}")

# Step 2: walk from the newest down until one actually holds the file. A brand
# new folder can be there before the file lands in it.
edition_year = file_name = None
for year in editions:
    listing = requests.get(f"{AREAS_BASE}/{year}/", timeout=120)
    if listing.status_code != 200:
        continue
    names = [h for h in HREF.findall(listing.text)
             if not h.startswith(("http", "/", "?"))]

    expected = f"AR_BR_RG_UF_RGINT_RGI_MUN_{year}.xls"
    match = next((n for n in names if n.lower() == expected.lower()), None)
    if match is None:
        # Some editions publish the same content under a slightly different name.
        match = next((n for n in names if n.lower().startswith("ar_br")
                      and n.lower().endswith((".xls", ".xlsx"))), None)
    if match:
        edition_year, file_name = year, match
        break
    print(f"  {year}: folder exists but has no areas file — trying the previous one")

if file_name is None:
    raise RuntimeError(
        f"No edition since {OLDEST_EDITION} actually contains the areas file")
print(f"Using edition {edition_year}: {file_name}")

# Step 3: download it.
source_dir = RAW_DIR / "territorial_areas"
source_dir.mkdir(parents=True, exist_ok=True)
destination = source_dir / file_name

if destination.exists():
    print(f"Already here: {file_name} ({destination.stat().st_size // 1024} KB)")
else:
    response = requests.get(f"{AREAS_BASE}/{edition_year}/{file_name}", timeout=300)
    response.raise_for_status()
    # geoftp can answer 200 with an HTML error page; a real edition is ~1 MB, so
    # anything tiny is that page and must not be saved under a .xls name.
    if len(response.content) < 10_000:
        raise RuntimeError(
            f"{file_name}: only {len(response.content)} bytes — probably an error page")
    destination.write_bytes(response.content)
    print(f"Downloaded: {file_name} ({destination.stat().st_size // 1024} KB)")

# Step 4: look at what arrived.
# The sheet and the area column both carry the year in their names, so we find
# them by prefix instead of rebuilding the string from `edition_year`.
from python_calamine import CalamineWorkbook

sheets = CalamineWorkbook.from_path(destination).sheet_names
print(f"\nSheets: {sheets}")

municipality_sheet = next((s for s in sheets if s.upper().startswith("AR_BR_MUN")), None)
if municipality_sheet is None:
    raise RuntimeError(f"No AR_BR_MUN* sheet in {file_name}; found {sheets}")

areas = pd.read_excel(destination, sheet_name=municipality_sheet, engine="calamine")
print(f"Municipality sheet: {municipality_sheet}")
print(f"Rows: {len(areas):,}  (the whole country — SP is filtered in 02_master)")
print(f"Columns: {list(areas.columns)}")

# CD_MUN comes as a float and the sheet has blank rows at the bottom, so this
# count is defensive on purpose. 645 is the number to expect for São Paulo.
sp_count = (areas["CD_MUN"].dropna().astype("int64").astype(str)
            .str.startswith("35").sum())
print(f"São Paulo municipalities present: {sp_count}")

areas.head(3)

Editions published: [2025, 2024, 2023, 2022, 2021, 2020]
Using edition 2025: AR_BR_RG_UF_RGINT_RGI_MUN_2025.xls
Already here: AR_BR_RG_UF_RGINT_RGI_MUN_2025.xls (903 KB)

Sheets: ['AR_BR_MUN_2025', 'AR_BR_RGINT_2025', 'AR_BR_RGI_2025', 'AR_BR_UF_2025', 'AR_BR_RG_2025', 'AR_BR_2025', 'Dicionário_e_Notas']
Municipality sheet: AR_BR_MUN_2025
Rows: 5,575  (the whole country — SP is filtered in 02_master)
Columns: ['CD_UF', 'NM_UF', 'NM_UF_SIGLA', 'CD_MUN', 'NM_MUN', 'AR_MUN_2025']
São Paulo municipalities present: 645


,CD_UF,NM_UF,NM_UF_SIGLA,CD_MUN,NM_MUN,AR_MUN_2025
0,11,Rondônia,RO,1100015.0,Alta Floresta D'Oeste,7067.127
1,11,Rondônia,RO,1100023.0,Ariquemes,4426.127
2,11,Rondônia,RO,1100031.0,Cabixi,1314.352


## Source 5 — SIH-SUS hospital admissions (DATASUS, FTP)

Every hospital admission paid for by the SUS, one file per month. We want it
because it measures something SINAN cannot: **severity**. SINAN records the case
that surveillance *recognised*; SIH-SUS records the case that *needed a bed*.
Mild dengue is never admitted, so this is a proxy for how bad an outbreak
actually got — and because admissions lag the wave, they are useful as a lagged
feature, a municipality's memory of recent hospital pressure.

`RD` stands for *reduzido*, the consolidated AIH (hospital admission
authorisation) record. It is filed under the patient's municipality of
**residence** (`MUNIC_RES`), even when the admission happened somewhere else —
so it also captures people who travelled to a reference hospital in a bigger
city.

Things that matter about this source:

- **This is the big one so far: about 2.2 GB.** Roughly 15 MB per month, one file
  per month since 2015. `SIH_START_YEAR` at the top of the cell shrinks it while
  you are developing — set it to, say, 2024 and you get a dozen files instead of
  a hundred and thirty. The cell lists the directory *first* and prints the exact
  total before downloading a single byte, so you can stop and think.
- **`.dbc` is not `.dbf`.** DATASUS compresses its DBF files with a scheme of its
  own that no ordinary tool opens. `pyreaddbc` expands `.dbc` → `.dbf` and
  `dbfread` reads that. Both are installed by the prerequisites cell; note that
  `pyreaddbc` compiles a C extension, so a compiler has to be available.
- **The DBF inside is latin-1, not UTF-8.** Read as UTF-8 it either raises or,
  worse, silently mangles the accents.
- **We list over FTP and never probe over HTTPS.** DATASUS answers `200 OK` over
  HTTPS for paths that do not exist, so a HEAD-style "does this month exist?"
  check lies without a sound. One FTP directory listing gives every name, size
  and publication date in a single round trip, and cannot produce a false
  positive. The sizes it reports are then used to verify each download.
- **A long FTP session drops.** A hundred-plus sequential retrievals on one
  connection will eventually hit a dead socket. The cell reconnects once per file
  before giving up on it.
- **`MUNIC_RES` is the 6-digit DATASUS code, not the 7-digit IBGE one** — the
  check digit is missing. Joining the two naively matches nothing at all. The
  6→7 lookup belongs to `02_master`.
- **`DIAG_PRINC` is space-padded** (`"A90 "`), and the arbovirus codes are A90 and
  A91 (dengue), A92.0 (chikungunya), A92.5 and A92.8 (zika), A95 (yellow fever).
  We download every admission and filter later — the raw file stays whole.
- **SUS only, and late.** Private-insurance admissions are simply not here, and
  AIH processing runs a few months behind SINAN. That lateness is why this source
  is not allowed to hold back the freshness limit of the database: when it is
  behind, it forward-fills instead of blocking everything else.
- **Personal columns are dropped on arrival.** Each RDSP file loses `CEP` (the
  patient's postal code) and the CPF fields `CPF_AUT` and `GESTOR_CPF` as soon as
  it is downloaded, and is kept as `RDSP{YY}{MM}.dbf.gz`: the DATASUS DBF without
  those columns, gzip-compressed, every other byte as published. Nothing in the
  project uses them, and the raw collection is archived publicly.

In [ ]:
# --- Source 5: SIH-SUS hospital admissions ---
import ftplib
import itertools
import re
import tempfile
from ftplib import FTP
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src" / "coleta"))
# Personal columns (postal code, CPFs, mother's date of birth) are dropped as
# soon as each file arrives; see src/coleta/colunas_pessoais.py.
from colunas_pessoais import abrir_dbf, arquivos_dbf, caminho_limpo, colunas_a_remover, limpar_dbc

SIH_HOST = "ftp.datasus.gov.br"
SIH_DIR = "/dissemin/publicos/SIHSUS/200801_/Dados"
SIH_UF = "SP"

# ---- The knob: how far back to go. -----------------------------------------
# 2015 is the real series we want (~130 files, ~2.2 GB, a long download).
# Raise it to 2024 or so while developing and the cell finishes in a minute.
SIH_START_YEAR = 2015

# RDSP2401.dbc  ->  RD + SP + year 24 + month 01
SIH_FILE = re.compile(rf"^RD{SIH_UF}(\d{{2}})(\d{{2}})\.dbc$", re.IGNORECASE)

source_dir = RAW_DIR / "sih_sus"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: list the directory once. This is deliberately FTP and not HTTPS —
# DATASUS answers 200 over HTTPS even for paths that do not exist, so probing
# month by month would happily "find" files that are not there. A listing
# cannot lie that way, and it hands us every file size for free.
ftp = FTP(SIH_HOST, timeout=300)
ftp.login()                        # DATASUS allows anonymous access
ftp.cwd(SIH_DIR)

listing = []
ftp.dir(listing.append)

# IIS listing lines look like:  "01-15-24  10:23AM   15277346 RDSP1501.dbc"
catalog = {}
for line in listing:
    parts = line.split()
    if len(parts) < 2:
        continue
    name = parts[-1]
    found = SIH_FILE.match(name)
    if not found:
        continue
    year, month = 2000 + int(found.group(1)), int(found.group(2))
    if not 1 <= month <= 12 or year < SIH_START_YEAR:
        continue
    catalog[name] = {"year": year, "month": month,
                     "bytes": int(parts[-2]) if parts[-2].isdigit() else 0}

if not catalog:
    raise RuntimeError(
        f"No RD{SIH_UF} file from {SIH_START_YEAR} on in {SIH_DIR} — "
        f"the directory listing returned {len(listing)} lines")

months = sorted((c["year"], c["month"]) for c in catalog.values())
total_bytes = sum(c["bytes"] for c in catalog.values())
print(f"{len(catalog)} monthly files available, "
      f"{months[0][0]}-{months[0][1]:02d} to {months[-1][0]}-{months[-1][1]:02d}")
print(f"Total size: {total_bytes / 1e9:.2f} GB "
      f"(set SIH_START_YEAR higher to shrink this)")

# Step 2: download them, skipping anything already complete on disk.
downloaded = skipped = 0
for index, name in enumerate(sorted(catalog), start=1):
    entry = catalog[name]
    destination = source_dir / name

    # A cleaned file on disk was whole when it was downloaded. A .dbc left by an
    # older run is checked against the listing size and then cleaned.
    if caminho_limpo(destination).exists():
        skipped += 1
        continue
    # The listing gave us the exact size, so "already here" can mean "already
    # here AND whole" rather than just "a file with that name exists".
    if destination.exists() and destination.stat().st_size == entry["bytes"]:
        limpar_dbc(destination)
        skipped += 1
        continue

    partial = destination.with_name(name + ".partial")
    for attempt in (1, 2):
        try:
            with open(partial, "wb") as out:
                ftp.retrbinary(f"RETR {name}", out.write, blocksize=1 << 20)
            break
        except (ftplib.Error, EOFError, OSError) as error:
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise RuntimeError(f"{name}: FTP failed twice ({error})")
            # A session this long loses its socket sooner or later. Reconnect
            # once and try this one file again before giving up on it.
            print(f"    connection dropped on {name} ({error}); reconnecting")
            ftp = FTP(SIH_HOST, timeout=300)
            ftp.login()
            ftp.cwd(SIH_DIR)

    got = partial.stat().st_size
    if got != entry["bytes"]:
        partial.unlink()
        raise RuntimeError(
            f"{name}: got {got:,} of the {entry['bytes']:,} bytes the listing promised")

    partial.rename(destination)
    limpar_dbc(destination)          # RDSPyymm.dbc -> RDSPyymm.dbf.gz without CEP and CPFs
    downloaded += 1
    print(f"  [{index:>3}/{len(catalog)}] {entry['year']}-{entry['month']:02d}  "
          f"{name}  {got / 1e6:.1f} MB")

ftp.quit()
print(f"\n{downloaded} downloaded, {skipped} already here")

# Step 3: look at what arrived, using the most recent month.
# The cleaned file is a gzipped .dbf; abrir_dbf expands it into a temp folder,
# so the expanded copy never lands in data/raw.
import dbfread

sample_name = sorted(catalog)[-1]
with abrir_dbf(caminho_limpo(source_dir / sample_name)) as sample_dbf:
    # latin1, not utf-8: DATASUS DBFs are latin-1 and reading them as UTF-8
    # either raises or, worse, quietly mangles every accent.
    table = dbfread.DBF(str(sample_dbf), encoding="latin1")
    print(f"\nSample file: {sample_name}")
    print(f"Rows: {len(table):,}")
    print(f"Columns ({len(table.field_names)}): {table.field_names}")
    sample = pd.DataFrame(list(itertools.islice(table, 3)))

# The three fields 02_master actually needs, under DATASUS's own names.
print("\nThe fields that matter, untranslated:")
print(sample[["MUNIC_RES", "DT_INTER", "DIAG_PRINC"]].to_string(index=False))

sample.head(3)

## Source 6 — Health network capacity and maternal mortality (DATASUS, FTP)

Two different databases living under one heading, because they answer the same
question from opposite ends: how much health system does a municipality have.

- **CNES/LT — hospital beds, monthly.** The national registry of health
  establishments; the `LT` module is the bed inventory. A municipality with no
  beds sends its severe cases somewhere else, which changes both how an outbreak
  is recorded and how it is survived.
- **SIM/DORES — death certificates, yearly.** We take it for maternal mortality
  (CID O00–O99), a standard proxy for the quality of primary care.

Things that matter about this source:

- **The two halves have very different lag, and that must not be averaged.**
  Consolidating death certificates nationally takes more than a year, while CNES
  comes out monthly. If you report a single "last period" for this source, a SIM
  that is 18 months behind makes the whole source look broken when the bed data
  is perfectly current. The cell prints **one limit per half**, deliberately.
- **`.dbc` and latin-1 again** — same DATASUS format as SIH-SUS, same
  `pyreaddbc` → `dbfread` path, same encoding trap.
- **Listing, not probing.** Same reason as SIH-SUS: DATASUS answers `200 OK` over
  HTTPS for files that do not exist, so we read each directory once over FTP and
  use the sizes it reports to verify what we download.
- **Two naming schemes.** `LTSP{YY}{MM}.dbc` is monthly, `DOSP{YYYY}.dbc` is
  yearly — the year is two digits in one and four in the other. Reusing one
  regex for both is how you silently collect nothing.
- **Neither half is allowed to hold back the database.** They are structural, not
  predictive: when they lag, `02_master` forward-fills them instead of trimming
  everything else back to match.
- **Personal columns are dropped on arrival** (CNES/LT only): `CPF_CNPJ`, the
  owner's CPF or CNPJ, leaves each LTSP file, which is kept as
  `LTSP{YY}{MM}.dbf.gz`, like SIH-SUS. The SIM files have no such column and stay
  as `.dbc`.

In [ ]:
# --- Source 6: Health network capacity and maternal mortality ---
import ftplib
import itertools
import re
import tempfile
from ftplib import FTP
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src" / "coleta"))
# Personal columns (postal code, CPFs, mother's date of birth) are dropped as
# soon as each file arrives; see src/coleta/colunas_pessoais.py.
from colunas_pessoais import abrir_dbf, arquivos_dbf, caminho_limpo, colunas_a_remover, limpar_dbc

HEALTH_HOST = "ftp.datasus.gov.br"
HEALTH_START_YEAR = 2015          # same knob idea as SIH-SUS: raise it to go faster

# Two datasets, two cadences, two naming schemes — one source.
HEALTH_DATASETS = [
    {"label": "beds",
     "what": "CNES/LT — hospital beds, monthly",
     "dir": "/dissemin/publicos/CNES/200508_/Dados/LT",
     "pattern": re.compile(r"^LTSP(\d{2})(\d{2})\.dbc$", re.IGNORECASE)},
    {"label": "deaths",
     "what": "SIM/DORES — death certificates, yearly",
     "dir": "/dissemin/publicos/SIM/CID10/DORES",
     "pattern": re.compile(r"^DOSP(\d{4})\.dbc$", re.IGNORECASE)},
]

source_dir = RAW_DIR / "health"
source_dir.mkdir(parents=True, exist_ok=True)

limits = {}          # the whole point: one freshness limit per half
newest_file = None

for dataset in HEALTH_DATASETS:
    print(f"\n{dataset['what']}")

    # Step 1: one listing per directory, over FTP.
    ftp = FTP(HEALTH_HOST, timeout=300)
    ftp.login()
    ftp.cwd(dataset["dir"])
    listing = []
    ftp.dir(listing.append)

    catalog = {}
    for line in listing:
        parts = line.split()
        if len(parts) < 2:
            continue
        found = dataset["pattern"].match(parts[-1])
        if not found:
            continue
        groups = found.groups()
        # LTSP2401 gives (24, 01); DOSP2024 gives (2024,) — the year is two
        # digits in one scheme and four in the other.
        if len(groups) == 2:
            year, month = 2000 + int(groups[0]), int(groups[1])
            if not 1 <= month <= 12:
                continue
        else:
            year, month = int(groups[0]), None
        if year < HEALTH_START_YEAR:
            continue
        catalog[parts[-1]] = {"year": year, "month": month,
                              "bytes": int(parts[-2]) if parts[-2].isdigit() else 0}

    if not catalog:
        raise RuntimeError(
            f"{dataset['label']}: nothing from {HEALTH_START_YEAR} on in {dataset['dir']}")

    periods = sorted((c["year"], c["month"] or 12) for c in catalog.values())
    limits[dataset["label"]] = periods[-1]
    total = sum(c["bytes"] for c in catalog.values())
    print(f"  {len(catalog)} files, up to {periods[-1][0]}-{periods[-1][1]:02d}, "
          f"{total / 1e6:.0f} MB")

    # Step 2: download, skipping what is already complete.
    downloaded = skipped = 0
    for name in sorted(catalog):
        entry = catalog[name]
        destination = source_dir / name
        limpa = bool(colunas_a_remover(name))     # CNES/LT yes, SIM no
        if limpa and caminho_limpo(destination).exists():
            skipped += 1
            continue
        if destination.exists() and destination.stat().st_size == entry["bytes"]:
            if limpa:
                limpar_dbc(destination)
            skipped += 1
            continue

        partial = destination.with_name(name + ".partial")
        for attempt in (1, 2):
            try:
                with open(partial, "wb") as out:
                    ftp.retrbinary(f"RETR {name}", out.write, blocksize=1 << 20)
                break
            except (ftplib.Error, EOFError, OSError) as error:
                partial.unlink(missing_ok=True)
                if attempt == 2:
                    raise RuntimeError(f"{name}: FTP failed twice ({error})")
                print(f"    connection dropped on {name}; reconnecting")
                ftp = FTP(HEALTH_HOST, timeout=300)
                ftp.login()
                ftp.cwd(dataset["dir"])

        got = partial.stat().st_size
        if got != entry["bytes"]:
            partial.unlink()
            raise RuntimeError(f"{name}: got {got:,} of {entry['bytes']:,} bytes")
        partial.rename(destination)
        if limpa:
            limpar_dbc(destination)      # LTSPyymm.dbc -> LTSPyymm.dbf.gz without CPF_CNPJ
        downloaded += 1

    ftp.quit()
    print(f"  {downloaded} downloaded, {skipped} already here")
    if dataset["label"] == "beds":
        newest_file = caminho_limpo(source_dir / sorted(catalog)[-1])

# Step 3: the two limits, side by side. This separation is the point: a SIM that
# is a year behind must not make the monthly bed data look stale too.
print("\nHow current each half is:")
for label, (year, month) in limits.items():
    print(f"  {label:7}: {year}-{month:02d}")

# Step 4: look at what arrived, using the most recent bed file.
with abrir_dbf(newest_file) as sample_dbf:
    table = dbfread.DBF(str(sample_dbf), encoding="latin1")     # latin1, not utf-8
    print(f"\nSample file: {newest_file.name}")
    print(f"Rows: {len(table):,}")
    print(f"Columns ({len(table.field_names)}): {table.field_names}")
    sample = pd.DataFrame(list(itertools.islice(table, 3)))

sample.head(3)

## Source 7 — Population and GDP (IBGE, SIDRA API)

Two annual municipal series, both from SIDRA:

| Table | Variable | What it is |
|---|---|---|
| 6579 | 9324 | resident population, *estimated* |
| 4709 | 93 | resident population, *counted* by the 2022 Census |
| 5938 | 37 | GDP at current prices (thousand R$) |

Population is not just another feature here — it is **the denominator of
incidence**, so it feeds straight into how outbreak labels are defined. A wrong
population silently rewrites the target variable.

Things that matter about this source:

- **The population series had been frozen at 2021 for five years, and nobody
  noticed.** The project assumed "IBGE only publishes estimates up to 2023" and
  forward-filled from there. Checked against the API in August 2026, the truth
  was both worse and better: the estimates series (6579) has **no 2022 and no
  2023 at all**, so those years were 2021 copied forward for all 645
  municipalities — while **2024 and 2025, assumed not to exist, were published
  with real values**. This is what a hard-coded assumption about a source costs.
- **The census year is not in the estimates series.** IBGE publishes the 2022
  *count* in its own table (4709), so we fetch it separately and stitch it in.
  Where the two overlap the census wins: a count beats a projection. After that,
  only **2023** genuinely has no published number — one gap instead of five.
- **GDP really is behind** (5938 stops at 2023), so `pib_per_capita` stays null
  for 2024–2025. There the *numerator* is missing, not the denominator — a
  different problem with a different fix, and worth not confusing.
- **The available years are read from each table's metadata, never hard-coded.**
  `/agregados/{table}/metadados` reports `periodicidade.inicio` and `.fim`, which
  is how a new year gets picked up on its own the moment IBGE publishes it.
- **Non-numeric codes mean null, never zero** — `-`, `..`, `...`, `X`, exactly as
  in the housing source.

Raw JSON goes to disk as provenance, and next to it two readable CSVs:
`sidra_population.csv` (estimates + census stitched together, with a `fonte`
column recording which is which) and `sidra_gdp.csv`.

In [ ]:
# --- Source 7: Population and GDP ---
import json

SIDRA_BASE = "https://servicodados.ibge.gov.br/api/v3/agregados"
IBGE_START_YEAR = 2015
NON_NUMERIC = {"-", "..", "...", "X", "x", ""}

# label -> (table, variable). Population arrives in two pieces on purpose: the
# estimates series simply does not contain the census year.
IBGE_QUERIES = {
    "population_estimate": (6579, 9324),
    "population_census":   (4709, 93),
    "gdp":                 (5938, 37),
}

source_dir = RAW_DIR / "ibge"
source_dir.mkdir(parents=True, exist_ok=True)

rows = []
for label, (table, variable) in IBGE_QUERIES.items():
    # Step 1: ask the table which years it actually has. Hard-coding a range here
    # is precisely how the population series sat frozen at 2021 for five years.
    meta = requests.get(f"{SIDRA_BASE}/{table}/metadados", timeout=120)
    meta.raise_for_status()
    periodicity = meta.json().get("periodicidade") or {}
    first, last = int(periodicity["inicio"]), int(periodicity["fim"])
    years = [y for y in range(max(first, IBGE_START_YEAR), last + 1)]
    if not years:
        raise RuntimeError(f"{label}: table {table} publishes nothing since {IBGE_START_YEAR}")
    print(f"{label:20} table {table}: metadata says {first}–{last}")

    # Step 2: one request for every year and every SP municipality at once.
    period = "|".join(str(y) for y in years)
    url = (f"{SIDRA_BASE}/{table}/periodos/{period}/variaveis/{variable}"
           f"?localidades=N6[N3[35]]")
    response = requests.get(url, timeout=300)
    response.raise_for_status()
    payload = response.json()
    if not payload or not payload[0].get("resultados"):
        raise RuntimeError(f"{label}: table {table} variable {variable} came back empty")

    destination = source_dir / f"sidra_{table}_{variable}.json"
    destination.write_text(json.dumps(payload, ensure_ascii=False, indent=2),
                           encoding="utf-8")

    # Step 3: flatten. The series really is missing years, so we record what came
    # back rather than what we expected to come back.
    got_years = set()
    for block in payload[0]["resultados"]:
        for item in block.get("series", []):
            for year, raw_value in (item.get("serie") or {}).items():
                got_years.add(int(year))
                rows.append({
                    "label": label,
                    "localidade_id": item["localidade"]["id"],
                    "localidade_nome": item["localidade"]["nome"],
                    "ano": int(year),
                    "valor": None if str(raw_value).strip() in NON_NUMERIC else raw_value,
                })
    missing = sorted(set(years) - got_years)
    print(f"{'':20} returned {min(got_years)}–{max(got_years)}"
          + (f", NOT published: {missing}" if missing else ", no gaps"))

ibge = pd.DataFrame(rows)
ibge["valor"] = pd.to_numeric(ibge["valor"], errors="coerce")

# Step 4: stitch the two population pieces into one series. Where a year exists
# in both, the census wins — a count beats a projection.
population = ibge[ibge["label"].str.startswith("population")].copy()
population["fonte"] = population["label"].map({
    "population_estimate": "estimativa", "population_census": "censo"})
population = (population.sort_values(["localidade_id", "ano", "fonte"])
              .drop_duplicates(["localidade_id", "ano"], keep="first")   # "censo" < "estimativa"
              .rename(columns={"valor": "pop_residente"})
              [["localidade_id", "localidade_nome", "ano", "pop_residente", "fonte"]]
              .reset_index(drop=True))
population.to_csv(source_dir / "sidra_population.csv", index=False)

gdp = (ibge[ibge["label"] == "gdp"]
       .rename(columns={"valor": "pib_mil_reais"})
       [["localidade_id", "localidade_nome", "ano", "pib_mil_reais"]]
       .sort_values(["localidade_id", "ano"]).reset_index(drop=True))
gdp.to_csv(source_dir / "sidra_gdp.csv", index=False)

# Step 5: look at what arrived. The year-by-year count is the check that matters:
# a year with 645 municipalities is real, a missing year is a real hole, and a
# year that is a copy of the one before is what we are here to avoid.
print(f"\nsidra_population.csv: {len(population):,} rows")
print(population.groupby(["ano", "fonte"])["localidade_id"].nunique().to_string())
print(f"\nsidra_gdp.csv: {len(gdp):,} rows, years "
      f"{int(gdp['ano'].min())}–{int(gdp['ano'].max())}")

population.head(3)

## Source 8 — Primary care coverage (e-Gestor APS, Ministry of Health)

The share of each municipality's population covered by the Family Health
Strategy. It matters twice over: primary care teams are who *detect* arbovirus
cases in the first place, so coverage shapes both the real burden and how much
of it ever gets reported.

The e-Gestor site is a single-page app, and underneath it calls a plain REST
endpoint that returns all 5,570 municipalities for one month:

```
https://relatorioaps-prd.saude.gov.br/cobertura/{ab|aps}
    ?unidadeGeografica=MUNICIPIO&nuCompInicio=YYYYMM&nuCompFim=YYYYMM
```

So no browser automation is needed — the endpoint was simply found in DevTools.

Things that matter about this source:

- **There is a methodology break in 2021, and the endpoint changes with it.**
  Until 2020 it is "Cobertura da Atenção Básica" at `/ab`, with the value in
  `pcCoberturaAb` as a *string in Brazilian number format*. From 2021 it is
  "Cobertura Potencial da APS" at `/aps`, with the value in `qtCobertura` as a
  real number. Different endpoint, different field, different type — and they
  are **not the same indicator**, so `02_master` keeps a flag marking which
  methodology each row came from instead of pretending it is one clean series.
- **An unpublished month answers `200 OK` with an empty list, not an error.** So
  "does this month exist?" has to look at the *content*. Checking the status code
  would tell you every month since 2015 exists, including next month's.
- **There is no metadata saying how far the data goes**, so the cell walks
  backwards from last month until something comes back with rows in it.
- The download is one JSON per month, saved with the endpoint in the file name so
  the `ab`/`aps` split stays visible on disk rather than being something you have
  to remember.

In [ ]:
# --- Source 8: Primary care coverage (ESF / APS) ---
import json
from datetime import timedelta

ESF_BASE = "https://relatorioaps-prd.saude.gov.br/cobertura"
ESF_START_YEAR = 2015
ESF_BREAK_YEAR = 2021        # from here on the endpoint is /aps, not /ab
ESF_PROBE_TRIES = 8          # how far back to look for the last published month

source_dir = RAW_DIR / "primary_care_coverage"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: find the most recent published month. There is no metadata for this,
# so we walk backwards from last month. The catch: a month that is not published
# answers 200 OK with an EMPTY LIST. Looking at the status code would "find"
# every month ever, so we look at the content instead.
probe = date.today().replace(day=1) - timedelta(days=1)     # last day of last month
latest = None
for _ in range(ESF_PROBE_TRIES):
    endpoint = "ab" if probe.year < ESF_BREAK_YEAR else "aps"
    competence = f"{probe.year}{probe.month:02d}"
    response = requests.get(
        f"{ESF_BASE}/{endpoint}?unidadeGeografica=MUNICIPIO"
        f"&nuCompInicio={competence}&nuCompFim={competence}", timeout=180)
    rows_here = []
    if response.status_code == 200:
        try:
            payload = response.json()
        except ValueError:
            payload = None
        # The API has answered as a bare list and as a wrapper object over time.
        if isinstance(payload, list):
            rows_here = payload
        elif isinstance(payload, dict):
            for key in ("dados", "content", "items", "resultado"):
                if isinstance(payload.get(key), list) and payload[key]:
                    rows_here = payload[key]
                    break
    if rows_here:
        latest = (probe.year, probe.month)
        print(f"Last published competence: {competence} ({len(rows_here):,} municipalities)")
        break
    print(f"  {competence}: nothing published yet")
    probe = probe.replace(day=1) - timedelta(days=1)

if latest is None:
    raise RuntimeError(
        f"No competence answered with data in the last {ESF_PROBE_TRIES} months")

# Step 2: every month from the start year up to that ceiling.
competences = []
year, month = ESF_START_YEAR, 1
while (year, month) <= latest:
    competences.append((year, month))
    year, month = (year + 1, 1) if month == 12 else (year, month + 1)
print(f"Downloading {len(competences)} monthly files "
      f"({ESF_START_YEAR}-01 to {latest[0]}-{latest[1]:02d})")

downloaded = skipped = 0
for year, month in competences:
    endpoint = "ab" if year < ESF_BREAK_YEAR else "aps"
    competence = f"{year}{month:02d}"
    # The endpoint goes in the file name so the 2021 methodology break stays
    # visible in the directory instead of being something you have to remember.
    destination = source_dir / f"cobertura_{endpoint}_{competence}.json"
    if destination.exists() and destination.stat().st_size > 1000:
        skipped += 1
        continue

    response = requests.get(
        f"{ESF_BASE}/{endpoint}?unidadeGeografica=MUNICIPIO"
        f"&nuCompInicio={competence}&nuCompFim={competence}", timeout=300)
    response.raise_for_status()
    payload = response.json()
    if not payload:
        raise RuntimeError(f"{competence}: empty answer inside a range we know is published")
    destination.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    downloaded += 1

print(f"{downloaded} downloaded, {skipped} already here")

# Step 3: look at what arrived — and specifically at the break. Printing the two
# column sets side by side is the fastest way to see that these really are two
# different indicators wearing the same name.
oldest = sorted(source_dir.glob("cobertura_ab_*.json"))
newest = sorted(source_dir.glob("cobertura_aps_*.json"))

for label, files in (("before 2021  (/ab )", oldest), ("2021 onwards (/aps)", newest)):
    if not files:
        continue
    payload = json.loads(files[-1].read_text(encoding="utf-8"))
    frame = pd.DataFrame(payload if isinstance(payload, list) else payload.get("dados", []))
    print(f"\n{label}  {files[-1].name}")
    print(f"  rows: {len(frame):,}")
    print(f"  columns: {list(frame.columns)}")

esf = pd.DataFrame(json.loads(newest[-1].read_text(encoding="utf-8")))
esf.head(3)

## Source 9 — Socioeconomic indicators: CAPAG and IDH-M

Two indicators of what a municipality can actually afford to do about an
outbreak.

- **CAPAG** (*Capacidade de Pagamento*, National Treasury) — a yearly rating of
  municipal fiscal health. A municipality rated C does not run a vector control
  programme the way an A does.
- **IDH-M** (*Atlas do Desenvolvimento Humano*, UNDP) — the municipal human
  development index, from the census.

**This source is expected to come back PARTIAL, and that is not a failure.** The
IDH-M cannot be collected automatically: the Atlas website is down, and the
surviving copy lives on Base dos Dados, behind BigQuery credentials. So the cell
downloads CAPAG and, for IDH-M, only *checks whether the file is already there*.
It does **not** raise — unlike every other source in this notebook, a missing
IDH-M is a documented manual step, not a broken download. It is 2010 census data
and never changes, so you do this once.

**Where the IDH-M file lives, and why not under `data/raw/`:** it is at
`data/lookup/idhm_2010_sp.csv`, next to the INMET coordinates. Everything else in
this notebook is written to a *dated* folder that changes on every run — a
hand-placed file there would have to be copied in again each time. Manual assets
belong somewhere stable and versioned, so `data/lookup/` is where they go.

**Provenance warning for the file currently there.** The original Atlas download
no longer exists anywhere in the old repository — there is no `IDHM_2010.csv`
on disk, only code that expected one. The values in `idhm_2010_sp.csv` were
recovered from `data/interim/socioeconomico.parquet` in the old project, which is
a *derived* artifact, not the source. It covers **642 of the 645** municipalities.
That is good enough to build the master today, but if the index ever needs
defending as a citation, download it properly from the link above and replace the
file.

Things that matter about CAPAG:

- **The file names follow no pattern whatsoever.** Real examples:
  `capag-municipios---2018.xlsx`, `tt20240515capag-municipios.xlsx`,
  `capag-municipios-posicao-2026-jun.xlsx`. Building a URL from a template is
  hopeless, which is why we go through the CKAN API and read the resource list.
- **Several years have more than one published position**, and picking the wrong
  one duplicates the key downstream. The tempting fix — sort by CKAN's
  `last_modified` — is wrong: in 2024 it pointed at the June position when
  October was already out. The reliable signal is the **date written in the
  resource name**, so that is what we sort by, falling back to CKAN's metadata
  only when the name has no date.
- **Only municipal resources**, since the same package also carries state-level
  files. We filter on the name and the format.

In [ ]:
# --- Source 9: Socioeconomic (CAPAG + IDH-M) ---
import re

CKAN_CAPAG = ("https://www.tesourotransparente.gov.br/ckan/api/3/action/package_show"
              "?id=9ff93162-409e-48b5-91d9-cf645a47fdfc")
CAPAG_FIRST_YEAR = 2018        # first edition ever published

# The IDH-M is a MANUAL ASSET, and it deliberately does NOT live under RAW_DIR:
# that folder is dated and changes on every run, so a hand-placed file there
# would have to be copied again each time. It sits in data/lookup/ instead,
# beside the INMET coordinates — versioned, collected once, never re-fetched.
IDHM_PATH = PROJECT_ROOT / "data" / "lookup" / "idhm_2010_sp.csv"
IDHM_SOURCE = ("https://basedosdados.org/dataset/cbfc7253-089b-44e2-8825-755e1419efc8"
               "?table=ec5fb3d1-fa98-4ab3-8a02-4b9950048a83")

YEAR_IN_NAME = re.compile(r"(20\d{2})")
DATE_IN_NAME = re.compile(r"(\d{2})/(\d{2})/(20\d{2})")

source_dir = RAW_DIR / "socioeconomic"
capag_dir = source_dir / "capag"
capag_dir.mkdir(parents=True, exist_ok=True)

# Step 1: ask CKAN what exists. There is no deducible URL pattern, so the
# resource list is the only reliable way in.
response = requests.get(CKAN_CAPAG, timeout=180)
response.raise_for_status()
package = response.json()
if not package.get("success"):
    raise RuntimeError("Treasury CKAN answered without success=true")

# Step 2: keep the most recent publication of each year.
chosen = {}
for resource in package["result"].get("resources", []):
    name = resource.get("name") or ""
    text = f"{name} {resource.get('url', '')}".lower()
    # The same package also carries state-level files.
    if "munic" not in text or (resource.get("format") or "").lower() not in ("xlsx", "xls", "csv"):
        continue
    found = YEAR_IN_NAME.search(name)
    if not found or int(found.group(1)) < CAPAG_FIRST_YEAR:
        continue
    year = int(found.group(1))

    # Recency key: the date written INSIDE the name, when there is one. CKAN's
    # own last_modified is not trustworthy here — in 2024 it pointed at the June
    # position while October was already published.
    date_found = DATE_IN_NAME.search(name)
    if date_found:
        day, month, name_year = date_found.groups()
        order = f"{name_year}-{month}-{day}"
    else:
        order = (resource.get("last_modified") or resource.get("created") or "")[:10]

    if year not in chosen or order > chosen[year]["order"]:
        chosen[year] = {"order": order, "url": resource.get("url") or "", "name": name}

if not chosen:
    raise RuntimeError(f"No municipal CAPAG resource from {CAPAG_FIRST_YEAR} on in the package")

# Step 3: download the chosen position of each year, under its original name.
for year in sorted(chosen):
    url = chosen[year]["url"]
    filename = url.rsplit("/", 1)[-1] or f"capag_municipios_{year}.xlsx"
    destination = capag_dir / filename
    if destination.exists() and destination.stat().st_size > 5000:
        print(f"  {year}: already here ({filename})")
        continue
    file_response = requests.get(url, timeout=600)
    file_response.raise_for_status()
    if len(file_response.content) < 5000:
        raise RuntimeError(f"CAPAG {year}: only {len(file_response.content)} bytes")
    destination.write_bytes(file_response.content)
    print(f"  {year}: {filename} ({destination.stat().st_size // 1024} KB)"
          f"  [position: {chosen[year]['order']}]")

# Step 4: IDH-M — check only. This is the one source in the notebook allowed to
# come back incomplete without raising: the Atlas site is down and the surviving
# copy needs BigQuery credentials, so this half is a documented manual step.
print("\nIDH-M (manual asset, not downloaded):")
if IDHM_PATH.exists():
    idhm = pd.read_csv(IDHM_PATH)
    print(f"  present: {IDHM_PATH.relative_to(PROJECT_ROOT)} "
          f"({len(idhm)} municipalities, "
          f"{idhm['idhm_2010'].min():.3f}–{idhm['idhm_2010'].max():.3f})")
    if len(idhm) < 645:
        print(f"  note: {645 - len(idhm)} of São Paulo's 645 municipalities have "
              f"no value here")
else:
    print(f"  MISSING: {IDHM_PATH}")
    print(f"  Put it there by hand — source: {IDHM_SOURCE}")
    print("  This is EXPECTED, not an error — 2010 census data that never changes.")

# Step 5: look at what arrived.
capag_files = sorted(capag_dir.glob("*.xls*"))
print(f"\nCAPAG files on disk: {len(capag_files)}")
for path in capag_files:
    print(f"  {path.name} ({path.stat().st_size // 1024} KB)")

newest_capag = capag_files[-1]
capag = pd.read_excel(newest_capag, engine="calamine")
print(f"\nSample: {newest_capag.name}")
print(f"Rows: {len(capag):,}")
print(f"Columns: {list(capag.columns)}")

capag.head(3)

## Source 10 — Land use and land cover (MapBiomas)

MapBiomas is a collaborative network of NGOs, universities and companies that
classifies every 30-metre pixel of Brazil from Landsat imagery with machine
learning, year by year since 1985, and publishes the areas aggregated per
municipality. Five classes matter for us: forest, farmland, non-vegetated area
(the urbanisation signal), water, and non-forest natural formation.

The link between land cover and arboviruses runs both ways — deforestation
pushes sylvatic yellow fever closer to people, while urbanisation is what
*Aedes aegypti* lives on — so this is one of the few sources that speaks to
dengue and yellow fever at once.

Things that matter about this source:

- **It is the only source here with no API and no FTP.** The spreadsheet
  (~75 MB) is hosted on Google Drive, and the Drive file id **changes with every
  new Collection** — one per year. The previous script had the id written into
  the code with a comment asking someone to update it by hand. Checked in August
  2026: the page was already publishing Collection 11 while the hard-coded id
  still pointed at 10.1. So the id is now read from the page on every run.
- **The collection number goes into the file name.** A new collection then lands
  next to the old one instead of silently overwriting it, and a change of
  collection becomes visible in the directory listing rather than being something
  you discover when your numbers move.
- **Google Drive does not hand over a large file on the first request.** It
  answers with an HTML "can't scan this for viruses" page and expects you to come
  back with a confirmation token. We detect that by the content type, pull the
  token out of the page, and ask again — a `requests.Session` carries the cookies
  along, which is what makes the second request work.
- **What Drive returns is a ZIP *containing* the spreadsheet, not the
  spreadsheet.** This one is nasty, and it broke the first real run: an `.xlsx`
  is itself a zip, so the file looks fine, saves fine under an `.xlsx` name, and
  then every reader fails with `File not found '_rels/.rels'` — an error that
  says nothing about the actual problem. The cell now looks inside the archive
  and, when it finds an `.xlsx` in there, unpacks that as the real file.
- **Reading the sample is slow.** It is a ~78 MB spreadsheet; opening it to show
  the sheet names and three rows takes a while. That is the price of the only
  format this source offers.

In [ ]:
# --- Source 10: MapBiomas land use ---
import re

MAPBIOMAS_PAGE = "https://brasil.mapbiomas.org/estatisticas/"
DRIVE_ID = re.compile(r"drive\.google\.com/uc\?id=([\w-]+)")
COLLECTION = re.compile(r"cole[çc][ãa]o\s*(\d+(?:\.\d+)?)", re.IGNORECASE)

source_dir = RAW_DIR / "mapbiomas"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: read the page and find today's Drive id and collection number. Both
# change with every new Collection, which is exactly why neither is written here.
page = requests.get(MAPBIOMAS_PAGE, timeout=180,
                    headers={"User-Agent": "Mozilla/5.0"})
page.raise_for_status()

ids = DRIVE_ID.findall(page.text)
if not ids:
    raise RuntimeError(
        "No Google Drive link on the MapBiomas page — they may have changed how "
        f"they publish. Check by hand at {MAPBIOMAS_PAGE}")
file_id = ids[0]

collections = COLLECTION.findall(page.text)
collection = max(collections, key=float) if collections else "unknown"
print(f"Collection {collection}, Drive id {file_id[:12]}...")

destination = source_dir / f"MAPBIOMAS_COVERAGE_COL_{collection.replace('.', '_')}.xlsx"

# Step 2: download from Drive. A file this big is not handed over on the first
# request: Drive answers with an HTML warning page and waits for a confirmation
# token. The Session is what carries the cookies between the two requests.
if destination.exists() and destination.stat().st_size > 1_000_000:
    print(f"Already here: {destination.name} "
          f"({destination.stat().st_size / 1e6:.0f} MB)")
else:
    session = requests.Session()
    response = session.get(f"https://drive.google.com/uc?export=download&id={file_id}",
                           stream=True, timeout=1800)
    response.raise_for_status()

    if "text/html" in response.headers.get("Content-Type", "").lower():
        body = response.text
        token = (re.search(r'name="confirm"\s+value="([\w-]+)"', body)
                 or re.search(r"confirm=([\w-]+)", body))
        params = {"id": file_id, "export": "download",
                  "confirm": token.group(1) if token else "t"}
        # Newer Drive pages also carry a uuid in the form; send it when present.
        uuid_found = re.search(r'name="uuid"\s+value="([\w-]+)"', body)
        if uuid_found:
            params["uuid"] = uuid_found.group(1)
        response = session.get("https://drive.usercontent.google.com/download",
                               params=params, stream=True, timeout=1800)
        response.raise_for_status()

    partial = destination.with_name(destination.name + ".partial")
    written = 0
    with open(partial, "wb") as out:
        for chunk in response.iter_content(chunk_size=1 << 20):
            out.write(chunk)
            written += len(chunk)

    # A small answer is the warning page again, not the spreadsheet. Saving it
    # under an .xlsx name would be the worst outcome: it looks like data.
    if written < 1_000_000:
        partial.unlink()
        raise RuntimeError(
            f"Drive returned only {written:,} bytes — that is the warning page, "
            f"not the spreadsheet")
    # What Drive hands over is a ZIP that CONTAINS the spreadsheet, not the
    # spreadsheet itself. Both are zip files underneath, so the extension lies
    # and any reader fails with a baffling "_rels/.rels not found". Unpack when
    # the archive turns out to hold an .xlsx of its own.
    import zipfile
    try:
        with zipfile.ZipFile(partial) as archive:
            inner = [n for n in archive.namelist() if n.lower().endswith(".xlsx")]
            if inner:
                destination.write_bytes(archive.read(inner[0]))
                partial.unlink()
                print(f"Downloaded and unpacked {inner[0]!r} "
                      f"({destination.stat().st_size / 1e6:.0f} MB)")
            else:
                partial.rename(destination)      # already a bare .xlsx
                print(f"Downloaded: {destination.name} ({written / 1e6:.0f} MB)")
    except zipfile.BadZipFile as error:
        partial.unlink()
        raise RuntimeError(
            f"Drive returned {written:,} bytes that are not a zip at all — "
            f"not a spreadsheet either ({error})")

# Step 3: if an older collection is still sitting here, say so out loud. Two
# collections in one folder is fine; silently reading the wrong one is not.
others = sorted(p.name for p in source_dir.glob("MAPBIOMAS_COVERAGE_COL_*.xlsx")
                if p.name != destination.name)
if others:
    print(f"NOTE: older collection(s) still in the folder: {', '.join(others)} — "
          f"make sure 02_master points at collection {collection}")

# Step 4: look at what arrived. This is slow: it is a 75 MB spreadsheet.
from python_calamine import CalamineWorkbook

sheets = CalamineWorkbook.from_path(destination).sheet_names
print(f"\nSheets: {sheets}")
mapbiomas = pd.read_excel(destination, sheet_name=sheets[0], engine="calamine", nrows=3)
print(f"First sheet: {sheets[0]}")
print(f"Columns ({len(mapbiomas.columns)}): {list(mapbiomas.columns)}")

mapbiomas.head(3)

## Source 11 — Water, sewage and drainage (SINISA)

SINISA is the national sanitation information system. Three of its indicators are
close to the mechanism of arbovirus transmission:

| Code | What it measures |
|---|---|
| `IAG0001` | population served by the water network (%) |
| `IES0001` | population served by the sewage collection network (%) |
| `IES0007` | households with sewage collected **and treated** (%) |
| `IAP0001` | paved public roads in the urban area (%) |

Unreliable piped water is the direct cause of stored water — drums, tanks,
cisterns — which is where *Aedes* breeds. Paving belongs to the drainage module,
not the management one, because it decides where standing water collects.

Things that matter about this source:

- **Mind the numbering: edition SINISA {N} publishes the data for year {N-1}.**
  What goes into the dataset is always the *reference year*, never the edition
  name. Getting this backwards shifts the whole series by one year.
- **The links follow no deducible pattern at all.** Surveyed in August 2026:
  water 2024 sits at `resultados-sinisa/SINISA_Resultados_Ref2024.zip`, water
  2023 at `sinisa/arquivos/SINISA_Resultados_Ref2023.zip` — a different
  directory — sewage 2024 at `SINISA_ESGOTO_Planilhas_2024.zip`, and sewage 2023
  behind an **intermediate page** that then links to the zip. So the cell reads
  the page, classifies each link by keyword, and follows one level of
  intermediate page when a link does not point straight at a file.
- **The keyword negatives are not optional.** `agua` is a substring of
  `aguaspluviais`, and the water package has a generic name
  (`SINISA_Resultados_Ref{YEAR}.zip`) with no "agua" in it at all — it is
  recognised by `resultados_ref`. Without the exclusions, water and rainwater
  drainage swap places.
- **Some modules are published as `.rar`**, which Python's standard library
  cannot read. That is what `7z` is for in the prerequisites cell.
- **This server drops connections mid-download, routinely.** The packages run to
  tens of megabytes and the first two real runs both died on
  `IncompleteRead(6,753,274 bytes read, ...)` partway through the rainwater
  module. So the download streams, compares what arrived against the
  `Content-Length` the server promised, and retries once — after which the same
  file came through fine both times.
- **`IES0007`, not `IES2004`.** The old README and schema said `IES2004`
  ("treated sewage relative to collected sewage") while the parser had always
  looked for `IES0007` ("population coverage"). They are different concepts —
  efficiency of a treatment plant versus coverage of the population — and we keep
  `IES0007`, which is both what the models were trained on and the one closer to
  the epidemiological mechanism of open sewage.

In [ ]:
# --- Source 11a: SINISA — find, download and unpack ---
import re
import shutil
import subprocess
import zipfile

SINISA_BASE = ("https://www.gov.br/cidades/pt-br/acesso-a-informacao/acoes-e-programas"
               "/saneamento/sinisa")
# Edition SINISA {N} carries the data of year {N-1}. The value we keep is the
# REFERENCE YEAR on the right, never the edition name in the URL.
SINISA_PAGES = {
    f"{SINISA_BASE}/resultados-sinisa/resultados-sinisa-2025": 2024,
    f"{SINISA_BASE}/resultados-sinisa": 2023,
}

# indicator -> (words that identify its module, words that rule it out).
# The negatives carry real weight: "agua" is a substring of "aguaspluviais", and
# the water package is named generically, so it is found by "resultados_ref".
SINISA_INDICATORS = {
    "IAG0001": (("resultados_ref", "agua"), ("pluviais", "esgoto", "residuos", "gestao")),
    "IES0001": (("esgoto",), ("pluviais",)),
    "IES0007": (("esgoto",), ("pluviais",)),
    "IAP0001": (("pluviais",), ()),
}
SINISA_EXTENSIONS = (".zip", ".xlsx", ".xls", ".rar")
HREF = re.compile(r'href="([^"#?]+)"', re.IGNORECASE)

source_dir = RAW_DIR / "sinisa"
source_dir.mkdir(parents=True, exist_ok=True)

sinisa_downloads = {}          # (reference year, indicator) -> file on disk

for page_url, reference_year in sorted(SINISA_PAGES.items(), key=lambda kv: -kv[1]):
    print(f"\nReference year {reference_year}  ({page_url.rsplit('/', 1)[-1]})")
    page = requests.get(page_url, timeout=180, headers={"User-Agent": "Mozilla/5.0"})
    page.raise_for_status()

    # Step 1: collect every sinisa link on the page, absolute.
    links = []
    for href in HREF.findall(page.text):
        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            href = "https://www.gov.br" + href
        elif not href.startswith("http"):
            href = page_url.rsplit("/", 1)[0] + "/" + href
        if "sinisa" in href.lower():
            links.append(href)

    # Step 2: classify the ones that point straight at a file, and remember the
    # pages that might hide one (sewage 2023 is behind exactly such a page).
    files, intermediate = {}, []
    for link in links:
        target = link.lower()
        if target.endswith(SINISA_EXTENSIONS):
            for code, (positive, negative) in SINISA_INDICATORS.items():
                normalised = target.replace("-", "_")
                if code in files or any(n in normalised for n in negative):
                    continue
                if any(p in normalised for p in positive):
                    files[code] = link
        elif "planilha" in target or "indicadores" in target:
            intermediate.append(link)

    # Step 3: follow one level of intermediate page for whatever is still missing.
    for extra_page in intermediate[:4]:
        if len(files) == len(SINISA_INDICATORS):
            break
        try:
            inner = requests.get(extra_page, timeout=180,
                                 headers={"User-Agent": "Mozilla/5.0"})
            inner.raise_for_status()
        except requests.RequestException:
            continue
        for href in HREF.findall(inner.text):
            link = href if href.startswith("http") else "https://www.gov.br" + href
            target = link.lower()
            if not target.endswith(SINISA_EXTENSIONS) or "retifica" in target:
                continue        # a loose "retificação" fixes one municipality
            for code, (positive, negative) in SINISA_INDICATORS.items():
                normalised = target.replace("-", "_")
                if code in files or any(n in normalised for n in negative):
                    continue
                if any(p in normalised for p in positive):
                    files[code] = link

    missing = sorted(set(SINISA_INDICATORS) - set(files))
    print(f"  found: {sorted(files)}" + (f" | NOT found: {missing}" if missing else ""))

    # Step 4: download each package under its original name.
    for code in sorted(files):
        url = files[code]
        filename = url.rsplit("/", 1)[-1]
        destination = source_dir / filename
        partial = destination.with_name(filename + ".partial")
        if not (destination.exists() and destination.stat().st_size > 5000):
            # Stream it and check the size. These packages run to tens of MB and
            # this server drops the connection often enough that reading
            # `.content` in one go fails on an ordinary day — which is exactly
            # what happened the first time this cell ran.
            for attempt in (1, 2):
                try:
                    with requests.get(url, timeout=900, stream=True,
                                      headers={"User-Agent": "Mozilla/5.0"}) as response:
                        response.raise_for_status()
                        expected = int(response.headers.get("Content-Length") or 0)
                        written = 0
                        with open(partial, "wb") as out:
                            for chunk in response.iter_content(chunk_size=1 << 20):
                                out.write(chunk)
                                written += len(chunk)
                    if expected and written != expected:
                        raise requests.RequestException(
                            f"truncated at {written:,} of {expected:,} bytes")
                    if written < 5000:
                        raise requests.RequestException(f"only {written} bytes")
                    partial.rename(destination)
                    break
                except requests.RequestException as error:
                    partial.unlink(missing_ok=True)
                    if attempt == 2:
                        raise RuntimeError(f"{filename}: download failed twice ({error})")
                    print(f"      {filename}: {error} — retrying once")
        sinisa_downloads[(reference_year, code)] = destination
        print(f"    {code}: {filename} ({destination.stat().st_size / 1e6:.1f} MB)")

# Step 5: unpack. Some modules come as .rar, which the standard library cannot
# read at all — that is what 7z is doing in the prerequisites cell.
for path in sorted(set(sinisa_downloads.values())):
    if path.suffix.lower() not in (".zip", ".rar"):
        continue
    unpacked = path.with_suffix("")
    if unpacked.exists():
        continue
    unpacked.mkdir(exist_ok=True)
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            archive.extractall(unpacked)
    else:
        if shutil.which("7z") is None:
            raise RuntimeError(
                f"{path.name} is a .rar and no extractor is installed.\n"
                f"  sudo apt install p7zip-full p7zip-rar")
        result = subprocess.run(["7z", "x", "-y", f"-o{unpacked}", str(path)],
                                capture_output=True)
        if result.returncode != 0:
            raise RuntimeError(f"7z failed on {path.name}: "
                               f"{result.stderr.decode('utf-8', 'replace')[:200]}")
    # Packages sometimes nest another archive inside.
    for inner in list(unpacked.rglob("*.zip")):
        with zipfile.ZipFile(inner) as archive:
            archive.extractall(inner.with_suffix(""))
    print(f"  unpacked {path.name}")

spreadsheets = sorted(source_dir.rglob("*.xlsx"))
print(f"\n{len(spreadsheets)} spreadsheet(s) unpacked. First few:")
for path in spreadsheets[:5]:
    print(f"  {path.relative_to(source_dir)}")

### Source 11 (continued) — pulling the four indicators out

The download above leaves a pile of spreadsheets; this second cell finds the four
indicators inside them. It is a separate cell because it fails for entirely
different reasons than a download does, and you want to see which of the two
broke.

**The header row is found, never assumed.** Its position changes between modules,
between editions, and again whenever the Ministry publishes a correction — a
correction inserts a "Data de retificação: ..." line and pushes everything down.
Measured across the two editions actually collected, the header lands on **three
different rows**: row 10 for the 2023 water and sewage sheets, row 11 for most of
the rest, and **row 12 for `SINISA_AGUA_Indicadores_Base Municipal_2024_Retificação.xlsx`**
— the corrected file, one row lower exactly as predicted. A fixed `skiprows`
would not raise; it would return the wrong column quietly. So we look for the
cell containing the indicator code and derive both the header row and the data
column from wherever that cell turns out to be.

In [ ]:
# --- Source 11b: SINISA — locate each indicator inside the spreadsheets ---
if not sinisa_downloads:
    raise RuntimeError(
        "Nothing to read: the cell above downloaded no package. Run it first, "
        "and if it failed, fix that failure rather than this one.")

sinisa_rows = []
sinisa_missing = []

for (reference_year, code), package in sorted(sinisa_downloads.items()):
    # The relevant sheet lives either in the file itself or in what we unpacked.
    if package.suffix.lower() in (".xlsx", ".xls"):
        candidates = [package]
    else:
        words = ("indicadores",) if code == "IAP0001" else ("indicadores", "municipal")
        candidates = [p for p in package.with_suffix("").rglob("*.xlsx")
                      if all(w in p.name.lower() for w in words)]

    extracted = None
    for candidate in candidates:
        workbook = pd.ExcelFile(candidate, engine="calamine")
        for sheet in workbook.sheet_names:
            if "nota" in sheet.lower() or "dicion" in sheet.lower():
                continue
            # Search the top of the sheet for the cell holding the code. Where it
            # sits tells us both the header row and the data column — which is
            # what survives a correction pushing everything one row down.
            top = pd.read_excel(candidate, sheet_name=sheet, engine="calamine",
                                header=None, nrows=25)
            position = None
            for row in range(len(top)):
                for column in range(top.shape[1]):
                    if str(top.iloc[row, column]).strip().upper() == code:
                        position = (row, column)
                        break
                if position:
                    break
            if not position:
                continue

            row, column = position
            full = pd.read_excel(candidate, sheet_name=sheet, engine="calamine",
                                 header=None, skiprows=row + 1)
            frame = pd.DataFrame({
                "cod_ibge": pd.to_numeric(full[0], errors="coerce"),
                "valor": pd.to_numeric(full[column], errors="coerce"),
            }).dropna(subset=["cod_ibge"])
            frame["cod_ibge"] = frame["cod_ibge"].astype("int64")
            frame = frame[frame["cod_ibge"].astype(str).str.len() == 7]
            if not frame.empty:
                frame["ano"] = reference_year
                frame["indicador"] = code
                extracted = frame
                print(f"  {reference_year} {code}: header on row {row + 1}, "
                      f"column {column} of {candidate.name!r} sheet {sheet!r} "
                      f"→ {len(frame):,} municipalities")
                break
        if extracted is not None:
            break

    if extracted is None:
        sinisa_missing.append(f"{reference_year}/{code}")
    else:
        sinisa_rows.append(extracted)

if not sinisa_rows:
    raise RuntimeError(
        f"None of the {len(sinisa_downloads)} package(s) contained a locatable "
        f"indicator. Not found: {sinisa_missing}")
if sinisa_missing:
    print(f"\nNOT located: {', '.join(sinisa_missing)}")

sinisa = pd.concat(sinisa_rows, ignore_index=True)
sinisa.to_csv(source_dir / "_indicators.csv", index=False)

# Look at what came out. The São Paulo count is the check that matters: 645 is
# the whole state, and a number far below it means the sheet was misread.
sp = sinisa[sinisa["cod_ibge"].astype(str).str.startswith("35")]
print(f"\n_indicators.csv: {len(sinisa):,} rows, columns {list(sinisa.columns)}")
print("\nSão Paulo municipalities with a value, by year and indicator:")
print(sp.dropna(subset=["valor"]).groupby(["ano", "indicador"])["cod_ibge"]
      .nunique().to_string())

sinisa.head(3)

## Source 12 — Municipal capacity and disaster history (IBGE MUNIC)

MUNIC is IBGE's survey of what municipal governments actually *have*: community
health agents, sanitary and epidemiological surveillance teams, endemic disease
control staff, risk mapping, a record of floods and landslides. It is the closest
thing we get to measuring whether a municipality can respond at all.

It is also the least systematic source in the pipeline, for three reasons the
cell has to absorb:

1. **The subject changes every edition — there is no series.** Each year IBGE
   surveys a different set of blocks. Surveyed in August 2026 for editions after
   2018:

   | Edition | Blocks surveyed | Variables we need |
   |---|---|---|
   | 2019 | HR, governance, public safety | none |
   | 2020 | risk management, environment | `Mgrd*`, `Mmam2612` |
   | 2021 | education, culture, sport, **health**, COVID | `Msau28`, `541`, `542`, `543` |
   | 2023 | social assistance, food security | none |
   | 2024 | housing, transport, agriculture | none |

   So the cell assumes nothing: it **scans** every edition for the codes we care
   about. A future edition that asks about health again is picked up on its own.

2. **The file name varies** — `Base_MUNIC_2020.xlsx`,
   `Base_MUNIC_2021_20240425.xlsx`, `Base_MUNIC_2018_xlsx_20201103.zip`. The name
   is discovered from the directory listing, never built from a template, and the
   zipped ones are unpacked to a predictable name.

3. **Codes get reused across editions with a different meaning.** This is the
   nasty one: `Msau28` in one edition is not guaranteed to be `Msau28` in the
   next. So finding the column is not enough — the cell also looks the code up in
   that edition's own dictionary sheet and checks the description still matches
   what we expect. A code found but *not* confirmed is reported rather than
   silently trusted.

Alongside the spreadsheets the cell writes `_editions.json`, mapping each edition
to the variables found in it and whether the dictionary confirmed each one. That
file is what lets `02_master` assemble a time series out of a survey that has no
series.

In [ ]:
# --- Source 12: IBGE MUNIC ---
import json
import re
import zipfile

MUNIC_BASE = "https://ftp.ibge.gov.br/Perfil_Municipios"
MUNIC_FIRST_YEAR = 2018
HREF = re.compile(r'href="([^"]+)"')

# code -> a phrase that must appear in that edition's dictionary description.
# The phrase is the whole point: it is what catches a code reused for a
# different question in a later edition.
MUNIC_VARIABLES = {
    "Msau28":  "agentes comunitários",
    "Msau541": "vigilância sanitária",
    "Msau542": "vigilância epidemiológica",
    "Msau543": "controle de endemias",
    "Mgrd01":  "seca",
    "Mgrd06":  "alagamento",
    "Mgrd07":  "erosiv",          # the dictionary says "processo erosivo acelerado"
    "Mgrd08":  "enchente",
    "Mgrd11":  "enxurrada",
    "Mgrd14":  "deslizamento",
    "Mgrd201": "mapeamento",
    "Mmam2612": "risco",
}

source_dir = RAW_DIR / "munic"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: which editions exist. One folder per year.
response = requests.get(f"{MUNIC_BASE}/", timeout=180)
response.raise_for_status()
editions = sorted({int(h.strip("/")) for h in HREF.findall(response.text)
                   if h.strip("/").isdigit() and int(h.strip("/")) >= MUNIC_FIRST_YEAR})
if not editions:
    raise RuntimeError(f"No MUNIC edition from {MUNIC_FIRST_YEAR} on at {MUNIC_BASE}/")
print(f"Editions available: {editions}")

munic_editions = {}
for year in editions:
    # Step 2: find the real file name — it is never the same twice.
    listing = requests.get(f"{MUNIC_BASE}/{year}/Base_de_Dados/", timeout=180)
    if listing.status_code != 200:
        print(f"  {year}: no Base_de_Dados directory")
        continue
    names = [h for h in HREF.findall(listing.text)
             if not h.startswith(("http", "/", "?"))]
    spreadsheets = [n for n in names if n.lower().endswith(".xlsx")]
    archives = [n for n in names if n.lower().endswith(".zip") and "xlsx" in n.lower()]
    remote = (spreadsheets or archives or [None])[0]
    if remote is None:
        print(f"  {year}: no base file in the directory")
        continue

    # Step 3: download and normalise the name, unpacking when it comes zipped.
    raw_path = source_dir / remote
    normalised = source_dir / f"Base_MUNIC_{year}.xlsx"
    if not normalised.exists():
        if not (raw_path.exists() and raw_path.stat().st_size > 10_000):
            file_response = requests.get(
                f"{MUNIC_BASE}/{year}/Base_de_Dados/{remote}", timeout=900)
            file_response.raise_for_status()
            if len(file_response.content) < 10_000:
                raise RuntimeError(f"MUNIC {year}: only {len(file_response.content)} bytes")
            raw_path.write_bytes(file_response.content)
        if raw_path.suffix.lower() == ".zip":
            with zipfile.ZipFile(raw_path) as archive:
                inner = [n for n in archive.namelist() if n.lower().endswith(".xlsx")]
                if not inner:
                    raise RuntimeError(f"MUNIC {year}: zip has no xlsx inside")
                normalised.write_bytes(archive.read(inner[0]))
        elif raw_path != normalised:
            normalised.write_bytes(raw_path.read_bytes())

    # Step 4: scan the edition for our codes. An unreadable sheet must not look
    # like "this edition does not ask about the subject", so it says so out loud.
    workbook = pd.ExcelFile(normalised, engine="calamine")
    wanted = {code.lower(): code for code in MUNIC_VARIABLES}
    found = {}
    for sheet in workbook.sheet_names:
        if "dicion" in sheet.lower():
            continue
        try:
            columns = pd.read_excel(normalised, sheet_name=sheet, engine="calamine",
                                    nrows=1).columns
        except Exception as error:
            print(f"    WARNING: sheet {sheet!r} of {normalised.name} is unreadable "
                  f"({type(error).__name__}); its variables will be missed")
            continue
        for column in columns:
            key = str(column).strip().lower()
            if key in wanted and wanted[key] not in found:
                found[wanted[key]] = sheet

    # Step 5: confirm each code against this edition's own dictionary.
    confirmed = {code: False for code in found}
    dictionary_sheets = [s for s in workbook.sheet_names if "dicion" in s.lower()]
    if dictionary_sheets and found:
        dictionary = pd.read_excel(normalised, sheet_name=dictionary_sheets[0],
                                   engine="calamine", header=None)
        text_by_row = [" | ".join(str(v) for v in dictionary.iloc[i].tolist()
                                  if str(v) != "nan").lower()
                       for i in range(len(dictionary))]
        for code in found:
            expected = MUNIC_VARIABLES[code].lower()
            confirmed[code] = any(code.lower() in line and expected in line
                                  for line in text_by_row)

    unconfirmed = sorted(c for c, ok in confirmed.items() if not ok)
    print(f"  {year}: {len(found)} variable(s) {sorted(found) if found else ''}"
          + (f" | NOT confirmed in the dictionary: {unconfirmed}" if unconfirmed else ""))

    munic_editions[str(year)] = {
        "file": normalised.name,
        "variables": {code: {"sheet": sheet, "dictionary_confirms": confirmed[code]}
                      for code, sheet in sorted(found.items())},
    }

(source_dir / "_editions.json").write_text(
    json.dumps(munic_editions, ensure_ascii=False, indent=2), encoding="utf-8")

# Step 6: look at what arrived.
with_data = {y: e for y, e in munic_editions.items() if e["variables"]}
if not with_data:
    raise RuntimeError("No edition contained any of the variables we need")
print(f"\n_editions.json written. Editions carrying our variables: {sorted(with_data)}")

newest_year = max(with_data)
newest = source_dir / with_data[newest_year]["file"]
sheet = next(iter(with_data[newest_year]["variables"].values()))["sheet"]
munic = pd.read_excel(newest, sheet_name=sheet, engine="calamine")
print(f"\nSample: {newest.name}, sheet {sheet!r}")
print(f"Rows: {len(munic):,}  (all 5,570 municipalities)")
print(f"Columns ({len(munic.columns)}): {list(munic.columns)[:15]} ...")

munic.head(3)

## Source 13 — SINAN: dengue, zika and chikungunya (DATASUS, FTP)

This is the target variable. Every notified case of the three urban arboviruses,
nationally, one file per disease per year:

```
.../SINAN/DADOS/FINAIS/{DENGBR|ZIKABR|CHIKBR}{YY}.dbc
.../SINAN/DADOS/PRELIM/{DENGBR|ZIKABR|CHIKBR}{YY}.dbc
```

The Ministry's REST API returns at most 20 records per call and cannot filter by
state, which makes it useless for bulk extraction. The FTP is the stable path.

Things that matter about this source:

- **FINAIS and PRELIM have opposite natures, and the cell treats them
  accordingly.** `FINAIS` is consolidated and does not change once published, so
  it is never re-downloaded. `PRELIM` is revised continuously — it is *precisely*
  where recent history keeps moving — so it is **always** re-downloaded, even
  when a file of the same name is already sitting there.
- **A year migrates from PRELIM to FINAIS when DATASUS consolidates it.** If you
  do not notice the migration, you stay on the preliminary version of that year
  forever. The cell records where each file came from in `_provenance.json` and
  re-fetches from FINAIS as soon as a previously preliminary year appears there.
- **Two freshness limits, not one, and they are printed separately.** Measured on
  ~2.5 million dengue cases in SP (FINAIS for 2023 and 2024), counting by month of
  notification: *confirmed* cases reach 93.6% completeness 60 days after the
  month ends in an epidemic year and 97.3% in a normal one, so 60 days is the cut
  for confirmed. For *notified* cases the incompleteness is much smaller — by
  definition every case of the month is in it, leaving only DATASUS's typing and
  publishing delay — so 30 days is used, conservatively, because `DT_DIGITA` does
  not exist in the DBC and the real delay cannot be measured from FINAIS. The gap
  matters: choosing confirmed over notified costs about a month of freshness.
- **The raw files are downloaded whole; the cut is metadata.** Partial months
  stay in the raw data on purpose — keeping them is what will let a future run
  measure stabilisation by comparing dated collections against each other.
  `02_master` applies the cut when it trims the base.
- **Same `.dbc` and latin-1 traps** as every other DATASUS source.

In [ ]:
# --- Source 13: SINAN (dengue, zika, chikungunya) ---
import calendar
import ftplib
import itertools
import json
import tempfile
from datetime import timedelta
from ftplib import FTP

SINAN_HOST = "ftp.datasus.gov.br"
SINAN_DIRS = {"PRELIM": "/dissemin/publicos/SINAN/DADOS/PRELIM",
              "FINAIS": "/dissemin/publicos/SINAN/DADOS/FINAIS"}
SINAN_PREFIXES = {"dengue": "DENGBR", "zika": "ZIKABR", "chikungunya": "CHIKBR"}
SINAN_START_YEAR = 2015           # raise it to shrink the download while developing

source_dir = RAW_DIR / "sinan"
source_dir.mkdir(parents=True, exist_ok=True)
provenance_path = source_dir / "_provenance.json"
provenance = (json.loads(provenance_path.read_text(encoding="utf-8"))
              if provenance_path.exists() else {})

# Step 1: list both directories. FINAIS wins where a file appears in both — a
# consolidated year supersedes its preliminary version.
catalog = {}
for dataset in ("PRELIM", "FINAIS"):
    ftp = FTP(SINAN_HOST, timeout=300)
    ftp.login()
    ftp.cwd(SINAN_DIRS[dataset])
    listing = []
    ftp.dir(listing.append)
    ftp.quit()
    for line in listing:
        parts = line.split()
        if len(parts) < 2:
            continue
        name = parts[-1]
        if name[:6].upper() not in {p for p in SINAN_PREFIXES.values()}:
            continue
        if not name.lower().endswith(".dbc"):
            continue
        year = 2000 + int(name[6:8])
        if year < SINAN_START_YEAR:
            continue
        catalog[name] = {"dataset": dataset, "year": year,
                         "bytes": int(parts[-2]) if parts[-2].isdigit() else 0}

if not catalog:
    raise RuntimeError(f"Neither SINAN directory listed any file from {SINAN_START_YEAR} on")

finais = sum(1 for v in catalog.values() if v["dataset"] == "FINAIS")
print(f"{len(catalog)} files: {finais} in FINAIS, {len(catalog) - finais} in PRELIM")
print(f"Total size: {sum(v['bytes'] for v in catalog.values()) / 1e9:.2f} GB")

# Step 2: download. The policy is the whole point of this cell.
ftp = FTP(SINAN_HOST, timeout=300)
ftp.login()
current_dir = None
counts = {"finais": 0, "prelim": 0, "unchanged": 0}

for name in sorted(catalog):
    entry = catalog[name]
    destination = source_dir / name
    dataset = entry["dataset"]

    # FINAIS never changes after publication, so a file we already took FROM
    # FINAIS is final. Anything else is re-fetched — including a file we have
    # from PRELIM whose year has since been consolidated.
    if (dataset == "FINAIS" and destination.exists()
            and provenance.get(name) == "FINAIS"
            and destination.stat().st_size == entry["bytes"]):
        counts["unchanged"] += 1
        continue

    if current_dir != SINAN_DIRS[dataset]:
        ftp.cwd(SINAN_DIRS[dataset])
        current_dir = SINAN_DIRS[dataset]

    partial = destination.with_name(name + ".partial")
    for attempt in (1, 2):
        try:
            with open(partial, "wb") as out:
                ftp.retrbinary(f"RETR {name}", out.write, blocksize=1 << 20)
            break
        except (ftplib.Error, EOFError, OSError) as error:
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise RuntimeError(f"{name}: FTP failed twice ({error})")
            print(f"    connection dropped on {name}; reconnecting")
            ftp = FTP(SINAN_HOST, timeout=300)
            ftp.login()
            ftp.cwd(SINAN_DIRS[dataset])
            current_dir = SINAN_DIRS[dataset]

    got = partial.stat().st_size
    if got != entry["bytes"]:
        partial.unlink()
        raise RuntimeError(f"{name}: got {got:,} of {entry['bytes']:,} bytes")
    partial.rename(destination)

    was = provenance.get(name)
    provenance[name] = dataset
    counts[dataset.lower()] += 1
    moved = "  (was PRELIM, now consolidated)" if was == "PRELIM" and dataset == "FINAIS" else ""
    print(f"  {name}  {dataset}  {got / 1e6:.0f} MB{moved}")

ftp.quit()
provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True),
                           encoding="utf-8")
print(f"\n{counts['finais']} from FINAIS, {counts['prelim']} from PRELIM, "
      f"{counts['unchanged']} already consolidated and left alone")

# Step 3: the two freshness limits. They are metadata, not a filter — the raw
# files above are whole, partial months included. 02_master applies the cut.
def last_month_older_than(days):
    """Most recent month whose end is already `days` in the past."""
    candidate = date.today().replace(day=1) - timedelta(days=1)
    while (date.today() - candidate).days < days:
        candidate = candidate.replace(day=1) - timedelta(days=1)
    return candidate.year, candidate.month

print("\nFreshness limits (measured on 2.5M dengue cases, by month of notification):")
for series, days in (("notified", 30), ("confirmed", 60)):
    year, month = last_month_older_than(days)
    print(f"  {series:9} ({days:>2}-day lag): up to {year}-{month:02d}")

# Step 4: look at what arrived — the newest dengue file.
newest = sorted(p for p in source_dir.glob("DENGBR*.dbc"))[-1]
with tempfile.TemporaryDirectory() as tmp:
    sample_dbf = Path(tmp) / f"{newest.stem}.dbf"
    dbc2dbf(str(newest), str(sample_dbf))
    table = dbfread.DBF(str(sample_dbf), encoding="latin1")
    print(f"\nSample file: {newest.name}  (from {provenance.get(newest.name, '?')})")
    print(f"Rows: {len(table):,}")
    print(f"Columns ({len(table.field_names)}): {table.field_names}")
    sample = pd.DataFrame(list(itertools.islice(table, 3)))

# The fields 02_master keys on, under SINAN's own names.
print("\nThe fields that matter, untranslated:")
print(sample[["ID_MUNICIP", "DT_NOTIFIC", "CLASSI_FIN"]].to_string(index=False))

sample.head(3)

## Source 14 — Weather (NASA POWER)

Temperature, rainfall, humidity, wind and pressure for each of the 645 São Paulo
municipalities, monthly. Together with SINAN this is the predictive core: the
mosquito's life cycle is governed by temperature, and its breeding sites by
rainfall, so this source and SINAN are the two that decide how far forward the
database can actually see.

**This is the slowest source in the notebook — around 27 minutes for all 645
municipalities**, one API call each plus a courtesy pause. Set
`NASA_MAX_MUNICIPALITIES` to a small number while you are developing; leave it at
`None` for the real thing.

Things that matter about this source:

- **We pull the daily endpoint and aggregate it ourselves, rather than using the
  monthly one.** Two reasons. First, the monthly endpoint publishes only the
  *mean* of rainfall, humidity, pressure and wind — minimum and maximum for those
  four simply do not exist unless we compute them. Second, measured in August
  2026, monthly was published only to 2025-12-31, almost eight months behind,
  while daily was three days from the present; using monthly would have frozen
  the weather far behind SINAN and made next-month prediction impossible.
- **The aggregation rule was verified against the API itself**, not assumed.
  Comparing June 2025 for São Paulo city, present in both endpoints:
  `T2M_MAX` is the **maximum** of the daily values (28.07 = 28.07), `T2M_MIN` the
  **minimum** (2.43 = 2.43), and everything else the **mean** (T2M 16.940 vs
  16.944; PS 92.910 vs 92.907). Divergences sit in the third decimal, which is
  the service's own rounding — so the stitched series has no artificial step in
  it.
- **A month only enters if every single day of it is present.** A partial month
  would quietly degrade the forecast, so incomplete months are dropped, not
  averaged over whatever days happened to arrive.
- **`-999` is the fill value, not a reading.** Left in, it would drag a monthly
  mean into nonsense. It is discarded before aggregating.
- **The coordinates come from a lookup file**, `data/lookup/municipios_sp_estacoes_inmet.xlsx`,
  which carries one latitude/longitude per municipality. The cell stops with a
  clear message if it is not there.

In [ ]:
# --- Source 14: NASA POWER weather ---
import calendar
import time
from collections import defaultdict
from datetime import timedelta
from urllib.parse import urlencode

# ---- The knob: how many municipalities to fetch. ---------------------------
# None = all 645, which takes about 27 minutes. Set it to 5 while developing.
NASA_MAX_MUNICIPALITIES = None

NASA_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
NASA_VARIABLES = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS10M", "PS"]
NASA_FILL_VALUE = -999.0
NASA_START_YEAR = 2015
NASA_PAUSE_S = 0.4              # courtesy pause between municipalities

# Verified against the API (June 2025, São Paulo city): the monthly T2M_MAX is
# the MAXIMUM of the daily values and T2M_MIN the MINIMUM; everything else is
# the MEAN. Keeping this correspondence is what stops the series from stepping.
NASA_AGGREGATION = {"T2M_MAX": max, "T2M_MIN": min}
# The four whose extremes exist only if we compute them: the monthly endpoint
# publishes just their mean.
NASA_EXTREMES = ("PRECTOTCORR", "RH2M", "PS", "WS10M")

lookup_path = PROJECT_ROOT / "data" / "lookup" / "municipios_sp_estacoes_inmet.xlsx"
if not lookup_path.exists():
    raise RuntimeError(
        f"Coordinates lookup not found at {lookup_path}.\n"
        f"It carries one latitude/longitude per SP municipality and is required here.")

source_dir = RAW_DIR / "nasa_power"
source_dir.mkdir(parents=True, exist_ok=True)

municipalities = pd.read_excel(lookup_path, engine="calamine")[
    ["Código Município Completo", "LATITUDE", "LONGITUDE"]]
municipalities.columns = ["cod_ibge", "lat", "lon"]
if NASA_MAX_MUNICIPALITIES:
    municipalities = municipalities.head(NASA_MAX_MUNICIPALITIES)
print(f"Municipalities to fetch: {len(municipalities)} "
      f"{'(LIMITED for development)' if NASA_MAX_MUNICIPALITIES else '(all of them)'}")

# The current month is never used: it is incomplete by definition.
end_day = date.today().replace(day=1) - timedelta(days=1)
period = (f"{NASA_START_YEAR}0101", end_day.strftime("%Y%m%d"))
print(f"Daily series from {period[0]} to {period[1]}")

records = []
started = time.time()
for position, row in enumerate(municipalities.itertuples(index=False), start=1):
    if position % 50 == 1:
        elapsed = time.time() - started
        print(f"  [{position:>3}/{len(municipalities)}]  {elapsed / 60:.1f} min elapsed")

    query = urlencode({
        "start": period[0], "end": period[1],
        "latitude": round(float(row.lat), 6), "longitude": round(float(row.lon), 6),
        "parameters": ",".join(NASA_VARIABLES),
        "community": "AG", "format": "JSON", "user": "arboviralicusp",
    })
    response = requests.get(f"{NASA_URL}?{query}", timeout=180)
    response.raise_for_status()
    parameter = response.json()["properties"]["parameter"]

    # Group the daily values by month, dropping the fill value as we go: -999 is
    # "no reading", and leaving it in would drag the monthly mean into nonsense.
    by_month = defaultdict(lambda: defaultdict(list))
    for variable, series in parameter.items():
        for day, value in series.items():
            if value != NASA_FILL_VALUE:
                by_month[day[:6]][variable].append(value)

    for competence, values in by_month.items():
        year, month = int(competence[:4]), int(competence[4:6])
        # Only complete months. A partial month degrades the forecast silently.
        if any(len(v) < calendar.monthrange(year, month)[1] for v in values.values()):
            continue
        record = {"cod_ibge": int(row.cod_ibge), "ano": year, "mes": month}
        for variable, daily in values.items():
            rule = NASA_AGGREGATION.get(variable, lambda v: sum(v) / len(v))
            record[variable.lower()] = rule(daily)
            if variable in NASA_EXTREMES:
                record[f"{variable.lower()}__min"] = min(daily)
                record[f"{variable.lower()}__max"] = max(daily)
        records.append(record)

    time.sleep(NASA_PAUSE_S)

if not records:
    raise RuntimeError("NASA POWER returned no complete month for any municipality")

weather = (pd.DataFrame(records)
           .sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True))
destination = source_dir / "nasa_power_municipios_sp.parquet"
weather.to_parquet(destination, index=False)

# Look at what arrived.
last_year = int(weather["ano"].max())
last_month = int(weather[weather["ano"] == last_year]["mes"].max())
print(f"\nWrote {destination.name} in {(time.time() - started) / 60:.1f} min")
print(f"Rows: {len(weather):,}  |  municipalities: {weather['cod_ibge'].nunique()}")
print(f"Series runs to {last_year}-{last_month:02d}")
print(f"Columns: {list(weather.columns)}")

weather.head(3)

## Source 15 — Yellow fever vaccination coverage (two systems, stitched)

The last source, and the most awkward: there is no single system holding the
series, because the Ministry of Health moved the whole thing mid-way.

- **1994–2022 — classic PNI TabNet.** A CGI form that generates a CSV on the
  server. This half is **frozen** — last updated in May 2023 — and will not move
  again.
- **2023–today — InfoMS / RNDS panel (Qlik Sense).** One export per year. The CSV
  is assembled *in the browser* from a Qlik hypercube and handed over through a
  temporary in-memory URL that does not exist on the server. There is no address
  to request, which is why this half needs a real browser.

**A methodological caveat that has to travel with the data:** the two halves come
from different systems (classic PNI versus RNDS) and are not strictly comparable.
Keep the break visible in whatever `02_master` builds — do not present them as one
smooth series.

Because the halves fail for completely different reasons, they are two separate
cells. The TabNet one is plain HTTP over a slow and occasionally flaky server;
the panel one drives Chrome and is the most fragile thing in this notebook. Both
retry before giving up.

Things that matter about the TabNet half:

- **The form is at `dhdat.exe` but the submission goes to `webtabx.exe`.**
  Posting back to the first URL just re-renders the form, with no error — you get
  an HTML page where you expected a CSV and nothing tells you why.
- **Everything is latin-1**, request body included, and the form's option values
  are read out of the page rather than hard-coded, so a renamed option surfaces
  as a clear failure instead of an empty result.

Things that matter about the panel half:

- **The panel's default state is BCG for the current year.** Exporting without
  applying the filters gives you a perfectly valid file about the wrong vaccine —
  which is exactly what the first attempt produced.
- **Qlik applies selections asynchronously.** Clicking export before the
  selection has landed exports the *previous* state, silently. So the cell waits
  until the on-screen labels actually say `Febre Amarela` and the year it asked
  for, and never exports anything it could not confirm.
- **Changing the year resets the vaccine back to BCG.** Confirmed in testing:
  the vaccine filter has to be re-applied *after* each year change, and the
  re-application is swallowed if you do not wait for the hypercube to reload.
- **Setting `value` from JavaScript is not enough** — without dispatching a
  `change` event, Qlik's observers never fire and the interface keeps the old
  selection.

**Three things found by actually running this:**

- **TLS verification stays on, and the fetch retries.** On 2026-09-09 this host
  briefly served an *expired* certificate and every client refused it, `curl`
  included — DATASUS renewing late, not a problem on your side. Their Let's
  Encrypt certificate is short-lived, so this will recur. The previous collector
  called plain `curl` with no `--insecure` anywhere and simply retried transient
  failures; the cell does the same, with waits of 5s, 15s and 45s. Nothing here
  ever disables certificate checking.
- **The panel retries too, and never skips a year quietly.** A Qlik selection
  that fails to land usually lands on the next try, so each year gets three
  attempts with growing waits. Only after all three does the year get abandoned —
  and then the cell prints an unmissable banner naming it, repeats the list of
  missing years at the end, and raises outright if nothing at all came back.
- **The panel export has a three-row header** — vaccine, then month, then the
  measure names, with data only from the fourth row. Read naively it gives you
  columns called `Febre Amarela.1`, `Febre Amarela.2`, and so on. The cell reads
  it raw, shows all three rows, and fails loudly if the first one does not say
  *Febre Amarela* — which is the BCG trap catching itself.

In [ ]:
# --- Source 15a: yellow fever vaccination — TabNet (1994-2022, frozen) ---
import re
import time
import urllib.parse

TABNET_FORM = "https://tabnet.datasus.gov.br/cgi/dhdat.exe?bd_pni/cpnibr.def"
# NOT the same URL: the form lives in dhdat.exe, but posting there just
# re-renders the form. The submission has to go to webtabx.exe.
TABNET_SUBMIT = "https://tabnet.datasus.gov.br/cgi/webtabx.exe?bd_pni/cpnibr.def"
TABNET_ROOT = "https://tabnet.datasus.gov.br/"

# TLS verification stays ON — the same thing the previous collector did, which
# called plain `curl` with no --insecure anywhere.
#
# Worth knowing, because it will happen again: on 2026-09-09 this host briefly
# served an EXPIRED certificate and every client refused it, curl included. It
# was DATASUS renewing late, not a problem with your machine — their Let's
# Encrypt certificate is short-lived and gets replaced every few months. So the
# fetch below retries with growing waits instead of dying on the first refusal,
# which is what the old collector did for any transient failure too.
TABNET_RETRY_WAITS_S = (5, 15, 45)

source_dir = RAW_DIR / "yellow_fever_vaccination"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: read the form. Everything here is latin-1, page and request body alike.
# It is a slow server and a 2 MB page, so the timeout is generous.
form = None
for attempt, wait in enumerate(TABNET_RETRY_WAITS_S, start=1):
    try:
        form = requests.get(TABNET_FORM, timeout=300)
        form.raise_for_status()
        break
    except (requests.exceptions.SSLError, requests.RequestException) as error:
        if attempt == len(TABNET_RETRY_WAITS_S):
            raise RuntimeError(
                f"TabNet did not answer after {attempt} attempts: {error}\n"
                "  If this is a certificate error, DATASUS is mid-renewal — wait "
                "and run again.\n"
                "  This half of the series is frozen at 2022, so fetching the CSV "
                "by hand once is an equally good answer.") from error
        print(f"  attempt {attempt} failed ({type(error).__name__}); "
              f"retrying in {wait}s", flush=True)
        time.sleep(wait)
html = form.content.decode("latin1")

# Step 2: read the option values out of the page instead of hard-coding them, so
# that a renamed option fails loudly rather than returning an empty table.
def option_starting_with(field, prefix):
    block = re.search(rf'<SELECT NAME="{field}".*?</SELECT>', html, re.S | re.I)
    if not block:
        return None
    for value, _ in re.findall(r'<option value="([^"]*)"[^>]*>([^<]*)', block.group(0)):
        if value.startswith(prefix):
            return value
    return None

line = option_starting_with("Linha", "Município")
column = option_starting_with("Coluna", "Ano")
measure = option_starting_with("Incremento", "Febre Amarela")
years_block = re.search(r'<SELECT NAME="PAno".*?</SELECT>', html, re.S | re.I)
years = ([v for v, _ in re.findall(r'<option value="([^"]*)"[^>]*>([^<]*)',
                                   years_block.group(0))] if years_block else [])

if not all((line, column, measure)) or not years:
    raise RuntimeError(
        "The TabNet form changed: expected option not found "
        f"(Linha={line!r}, Coluna={column!r}, Incremento={measure!r}, "
        f"{len(years)} year options)")
print(f"Form read: Linha={line!r}, Coluna={column!r}, Incremento={measure!r}")

# Step 3: submit. One "Arquivos" per request is allowed, and the most recent one
# already carries the whole series across the columns, because the column IS Ano.
body = urllib.parse.urlencode([
    ("Linha", line), ("Coluna", column), ("Incremento", measure),
    ("Arquivos", years[0]),
    ("SRegião", "TODAS_AS_CATEGORIAS__"), ("SUnidade", "TODAS_AS_CATEGORIAS__"),
    ("SMunicípio", "TODAS_AS_CATEGORIAS__"), ("SCapital", "TODAS_AS_CATEGORIAS__"),
    ("formato", "table"), ("mostre", "Mostra"),
], encoding="latin1")

answer = requests.post(
    TABNET_SUBMIT, data=body.encode("latin1"), timeout=600,
    headers={"Content-Type": "application/x-www-form-urlencoded",
             "User-Agent": "Mozilla/5.0"})
answer.raise_for_status()

# Step 4: the response is an HTML page carrying a link to a CSV built on the
# server. No link means the query came back as a page, not a table.
found = re.search(r'href="?([^"> ]*\.csv)', answer.content.decode("latin1"), re.I)
if not found:
    raise RuntimeError("TabNet answered without a CSV link — the query was rejected")

csv_url = urllib.parse.urljoin(TABNET_ROOT, found.group(1))
destination = source_dir / "cob_vac_fa_tabnet_1994_2022.csv"
csv_response = requests.get(csv_url, timeout=600)
csv_response.raise_for_status()
if len(csv_response.content) < 10_000:
    raise RuntimeError(f"TabNet CSV is only {len(csv_response.content)} bytes")
destination.write_bytes(csv_response.content)
print(f"Downloaded: {destination.name} ({destination.stat().st_size // 1024} KB)")

# Step 5: look at what arrived. TabNet CSVs are latin-1 and semicolon separated,
# with a title block before the table and notes after it. How many lines that
# block has depends on the query, so we FIND the header row rather than assume
# it — the same principle as the SINISA spreadsheets.
lines = destination.read_text(encoding="latin1").splitlines()
print("First lines of the file, as delivered:")
for line in lines[:5]:
    print(f"  {line[:100]}")

header_row = next((i for i, line in enumerate(lines)
                   if line.lstrip('"').lower().startswith("munic")), None)
if header_row is None:
    raise RuntimeError("No header row starting with 'Munic' in the TabNet CSV")

tabnet = pd.read_csv(destination, sep=";", encoding="latin1", skiprows=header_row,
                     engine="python", on_bad_lines="skip")
print(f"\nHeader found on line {header_row + 1}")
print(f"Rows: {len(tabnet):,}")
print(f"Columns: {list(tabnet.columns)}")

tabnet.head(3)

In [ ]:
# --- Source 15b: yellow fever vaccination — InfoMS panel (2023+, Selenium) ---
import shutil
import tempfile
import time

PANEL_URL = ("https://infoms.saude.gov.br/extensions/"
             "SEIDIGI_DEMAS_VACINACAO_CALENDARIO_NACIONAL_COBERTURA_RESIDENCIA/index.html")
PANEL_FIRST_YEAR = 2023          # the first year beyond TabNet's reach
PANEL_TAB_DATA = "aba2-tab"
PANEL_EXPORT_BUTTON = "exportar-dados-linha-municipios"
PANEL_LABEL_YEAR = "interaction-Ano-floating"
PANEL_LABEL_VACCINE = "interaction-Imunobiológico-floating"
PANEL_VACCINE = "Febre Amarela"
PANEL_LOAD_S = 30                # Qlik takes this long to build the hypercube
PANEL_DOWNLOAD_S = 90
PANEL_HEADLESS = True            # set False to watch it work, which helps when it breaks

# Setting `value` from script does NOT wake Qlik's observers: without dispatching
# a change event the interface keeps the old selection and the export comes out
# with the previous state.
JS_SELECT = """
const [index, target] = arguments;
const s = document.querySelectorAll('select')[index];
if (!s) return 'select missing';
const option = Array.from(s.options).find(o => o.text.trim() === target);
if (!option) return 'option missing';
s.value = option.value;
s.dispatchEvent(new Event('change', {bubbles: true}));
return 'ok';
"""
JS_YEARS = r"""
const s = document.querySelectorAll('select')[0];
return s ? Array.from(s.options).map(o => o.text.trim()).filter(t => /^\d{4}$/.test(t)) : [];
"""
JS_LABELS = """
const text = id => (document.getElementById(id) || {}).textContent || '';
return [text(arguments[0]), text(arguments[1])];
"""

try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
except ImportError as error:
    raise RuntimeError("selenium is not installed. Run: pip install selenium") from error

download_dir = Path(tempfile.mkdtemp(prefix="panel_fa_"))
options = Options()
if PANEL_HEADLESS:
    options.add_argument("--headless=new")
for argument in ("--no-sandbox", "--disable-dev-shm-usage", "--window-size=1920,1080"):
    options.add_argument(argument)
options.add_experimental_option("prefs", {
    "download.default_directory": str(download_dir),
    "download.prompt_for_download": False,
})

browser = webdriver.Chrome(options=options)
collected = []
panel_failed = []          # years that must be re-collected — never silent
PANEL_ATTEMPTS = 3
try:
    print(f"Opening the panel (Qlik takes about {PANEL_LOAD_S}s to build)")
    browser.get(PANEL_URL)
    time.sleep(PANEL_LOAD_S)

    available = [int(y) for y in browser.execute_script(JS_YEARS)]
    targets = sorted(y for y in available if y >= PANEL_FIRST_YEAR)
    if not targets:
        raise RuntimeError(f"No year >= {PANEL_FIRST_YEAR} in the panel (found {available})")
    print(f"Years in the panel: {available} | collecting {targets}")

    for year in targets:
        # The panel is flaky in a recoverable way: a selection that does not land
        # usually lands on the next try. So we try the whole selection dance up
        # to PANEL_ATTEMPTS times with growing waits, and only give up after.
        confirmed = False
        year_label = vaccine_label = "(not read)"
        for attempt in range(1, PANEL_ATTEMPTS + 1):
            if attempt > 1:
                wait = 15 * (attempt - 1)          # 15s, then 30s
                print(f"  {year}: attempt {attempt} of {PANEL_ATTEMPTS} "
                      f"after {wait}s")
                time.sleep(wait)

            if browser.execute_script(JS_SELECT, 0, str(year)) != "ok":
                print(f"  {year}: year filter did not take on attempt {attempt}")
                continue

            # Changing the year reloads the hypercube and RESETS the vaccine back
            # to BCG. Re-applying without waiting for the reload gets swallowed,
            # which is how the first ever run exported BCG instead of yellow fever.
            time.sleep(10)
            for _ in range(3):
                browser.execute_script(JS_SELECT, 1, PANEL_VACCINE)
                time.sleep(6)
                _, vaccine_label = browser.execute_script(
                    JS_LABELS, PANEL_LABEL_YEAR, PANEL_LABEL_VACCINE)
                if PANEL_VACCINE.lower() in vaccine_label.lower():
                    break

            # Only proceed once the on-screen labels agree with what we asked for.
            # Qlik applies selections asynchronously and exports the OLD state
            # without complaining if you click too early.
            deadline = time.time() + 60
            while time.time() < deadline:
                year_label, vaccine_label = browser.execute_script(
                    JS_LABELS, PANEL_LABEL_YEAR, PANEL_LABEL_VACCINE)
                if (str(year) in year_label
                        and PANEL_VACCINE.lower() in vaccine_label.lower()):
                    confirmed = True
                    break
                time.sleep(2)
            if confirmed:
                break
            print(f"  {year}: attempt {attempt} did not confirm "
                  f"('{year_label}' / '{vaccine_label}')")

        if not confirmed:
            panel_failed.append(year)
            print("\n" + "!" * 74)
            print(f"!!  YEAR {year} NOT COLLECTED — the Qlik selection never "
                  f"confirmed after {PANEL_ATTEMPTS} attempts.")
            print(f"!!  Last labels seen: '{year_label}' / '{vaccine_label}'")
            print(f"!!  This year is MISSING from the data. Re-run this cell; if "
                  f"it keeps failing,")
            print(f"!!  set PANEL_HEADLESS = False and watch what the panel does.")
            print("!" * 74 + "\n", flush=True)
            continue

        browser.execute_script(f"document.getElementById('{PANEL_TAB_DATA}').click();")
        time.sleep(8)
        buttons = browser.find_elements("id", PANEL_EXPORT_BUTTON)
        if not buttons:
            raise RuntimeError(f"Export button {PANEL_EXPORT_BUTTON!r} is gone — panel changed")

        seen = {p.name for p in download_dir.iterdir()}
        browser.execute_script("arguments[0].click();", buttons[0])

        # The file is assembled in the browser, so we wait for it to appear.
        arrived = None
        deadline = time.time() + PANEL_DOWNLOAD_S
        while time.time() < deadline:
            fresh = [p for p in download_dir.iterdir()
                     if p.name not in seen and p.suffix.lower() in (".csv", ".xlsx")
                     and not p.name.endswith(".crdownload")]
            if fresh:
                arrived = fresh[0]
                break
            time.sleep(2)
        if arrived is None:
            panel_failed.append(year)
            print("\n" + "!" * 74)
            print(f"!!  YEAR {year} NOT COLLECTED — the selection was right but the "
                  f"download never")
            print(f"!!  finished within {PANEL_DOWNLOAD_S}s. This year is MISSING. "
                  f"Re-run this cell.")
            print("!" * 74 + "\n", flush=True)
            continue

        destination = source_dir / f"cob_vac_painel_{year}{arrived.suffix}"
        shutil.copy2(arrived, destination)
        collected.append(destination)
        print(f"  {year}: {destination.name} ({destination.stat().st_size // 1024} KB)")
finally:
    browser.quit()
    shutil.rmtree(download_dir, ignore_errors=True)

if not collected:
    raise RuntimeError(
        f"The panel exported nothing. Years attempted: {targets}; all failed.")

# Never let a missing year pass quietly: it has to be the last thing you read.
if panel_failed:
    print("!" * 74)
    print(f"!!  INCOMPLETE COLLECTION — missing year(s): {sorted(set(panel_failed))}")
    print(f"!!  Collected: {sorted(int(p.stem.split('_')[-1]) for p in collected)}")
    print(f"!!  Re-run this cell to pick up what is missing before building the master.")
    print("!" * 74)
else:
    print(f"All {len(collected)} year(s) collected, none missing.")

# Look at what arrived. The export has a THREE-ROW header: the vaccine on the
# first row, the month on the second, and the measure names (Cobertura Vacinal,
# Doses Aplicadas, População) only on the third, with data from the fourth. Let
# pandas guess and you get columns called "Febre Amarela.1", "Febre Amarela.2",
# which is neither wrong nor useful — so we read it raw and show the shape.
newest = collected[-1]
panel_raw = (pd.read_csv(newest, sep=";", encoding="latin1", header=None,
                         engine="python", on_bad_lines="skip")
             if newest.suffix.lower() == ".csv"
             else pd.read_excel(newest, engine="calamine", header=None))

print(f"\nSample: {newest.name}")
print(f"Rows: {len(panel_raw):,} (3 of them are header)")
print("\nThe three header rows, first columns:")
for position in range(3):
    print(f"  row {position}: {[str(v)[:20] for v in panel_raw.iloc[position].tolist()[:6]]}")

# The check that matters: the panel's default is BCG, so if the first row does
# not say Febre Amarela the filter did not hold and this file is another vaccine.
vaccines = {str(v) for v in panel_raw.iloc[0].tolist() if str(v) != "nan"}
if not any(PANEL_VACCINE.lower() in v.lower() for v in vaccines):
    raise RuntimeError(f"Wrong vaccine in {newest.name}: header row says {vaccines}")
print(f"\nVaccine in the header: {sorted(vaccines)}  — correct")

panel_raw.head(3)

## Source 16 — Live births (SINASC, DATASUS FTP)

The denominator. On its own a count of births says little about arboviruses;
what it does is turn Source 6's raw count of maternal deaths into the
**maternal mortality ratio** — deaths per 100,000 live births — which is the
standard indicator and a real proxy for the quality of primary care. Without it,
"maternal mortality" is just a count that mostly reflects how many people live in
the municipality.

```
.../SINASC/1996_/Dados/DNRES/DNSP{YYYY}.dbc
```

Things that matter about this source:

- **The series ends in 2024, and that is exactly right.** SINASC stops at 2024
  and so does SIM/DORES in Source 6, because both take about two years to
  consolidate nationally. Numerator and denominator therefore cover **the same
  window**, and the ratio is never computed by dividing 2024 deaths by 2023
  births. If one of the two ever runs ahead of the other, that is the bug to look
  for.
- **`CODMUNRES` is the mother's municipality of residence — and that is the one
  to use.** SIM aggregates maternal deaths by residence too, so both sides of the
  ratio sit on the same population. The file also carries `CODMUNNASC`, where the
  birth physically happened, and the two genuinely differ: in 2024, 576 births to
  São Paulo mothers took place in other states. Using the wrong one would put the
  birth in Minas Gerais and the death in São Paulo.
- **`DTNASC` is `DDMMYYYY`, not `YYYYMMDD`.** This is the opposite of `DT_NOTIFIC`
  in SINAN and `DT_INTER` in SIH-SUS. Reusing their date parser here would read
  `06062024` as the year 601. **We sidestep it entirely: the year comes from the
  file name.** That is safe because each file holds only its own year — checked
  on `DNSP2024`, all 470,887 records are from 2024, with none of the
  spill-over that makes SINAN's files messy.
- **`DNRES` is already filtered to residents**, so unlike SINAN there is no UF
  column to filter on: `CODMUNRES` came back 100% prefix `35` across the whole
  file. What does need dropping is `350000`, DATASUS's "municipality unknown"
  code — 4 records in 2024, which is 0.0008% and moves nothing.
- **Everything else is the familiar DATASUS pattern**: `.dbc` expanded with
  `pyreaddbc`, read as **latin-1**, and `CODMUNRES` is the 6-digit code needing
  the same 6→7 lookup as SINAN and SIH-SUS. Coverage is complete: all **645**
  municipalities had at least one birth in 2024, from the capital's 120,789 down
  to four in the smallest.
- **It is cheap.** Ten files, about 299 MB, roughly two to four minutes — the
  least expensive source in the notebook.

This cell only collects. The division that turns deaths and births into a ratio
belongs to `02_master`, along with renaming the column to say what it finally is.
- **Personal columns are dropped on arrival**: `DTNASCMAE`, the mother's date of
  birth, leaves each file, which is kept as `DNSP{YYYY}.dbf.gz`, like SIH-SUS.

In [ ]:
# --- Source 16: Live births (SINASC) ---
import ftplib
import re
import tempfile
from collections import Counter
from ftplib import FTP

import dbfread
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src" / "coleta"))
# Personal columns (postal code, CPFs, mother's date of birth) are dropped as
# soon as each file arrives; see src/coleta/colunas_pessoais.py.
from colunas_pessoais import abrir_dbf, arquivos_dbf, caminho_limpo, colunas_a_remover, limpar_dbc

SINASC_HOST = "ftp.datasus.gov.br"
SINASC_DIR = "/dissemin/publicos/SINASC/1996_/Dados/DNRES"
SINASC_START_YEAR = 2015          # matches the project series; SINASC itself goes back to 1996

# DNSP2024.dbc -> DN + SP + four-digit year. Note the year has FOUR digits here,
# unlike SIH-SUS and SINAN, which use two.
SINASC_FILE = re.compile(r"^DNSP(\d{4})\.dbc$", re.IGNORECASE)

source_dir = RAW_DIR / "sinasc"
source_dir.mkdir(parents=True, exist_ok=True)

# Step 1: list the directory once, over FTP, for the same reason as every other
# DATASUS source — the listing cannot invent a file that is not there, and it
# hands us each size to verify the download against.
ftp = FTP(SINASC_HOST, timeout=300)
ftp.login()
ftp.cwd(SINASC_DIR)
listing = []
ftp.dir(listing.append)

catalog = {}
for line in listing:
    parts = line.split()
    if len(parts) < 2:
        continue
    found = SINASC_FILE.match(parts[-1])
    if not found:
        continue
    year = int(found.group(1))
    if year < SINASC_START_YEAR:
        continue
    catalog[parts[-1]] = {"year": year,
                          "bytes": int(parts[-2]) if parts[-2].isdigit() else 0}

if not catalog:
    raise RuntimeError(
        f"No DNSP file from {SINASC_START_YEAR} on in {SINASC_DIR}")

years = sorted(c["year"] for c in catalog.values())
print(f"{len(catalog)} yearly files, {years[0]} to {years[-1]}")
print(f"Total size: {sum(c['bytes'] for c in catalog.values()) / 1e6:.0f} MB")
print(f"Series ends in {years[-1]} — SIM/DORES in Source 6 should end in the same "
      f"year, so the ratio stays inside one window")

# Step 2: download, skipping anything already complete.
downloaded = skipped = 0
for name in sorted(catalog):
    entry = catalog[name]
    destination = source_dir / name
    if caminho_limpo(destination).exists():
        skipped += 1
        continue
    if destination.exists() and destination.stat().st_size == entry["bytes"]:
        limpar_dbc(destination)
        skipped += 1
        continue

    partial = destination.with_name(name + ".partial")
    for attempt in (1, 2):
        try:
            with open(partial, "wb") as out:
                ftp.retrbinary(f"RETR {name}", out.write, blocksize=1 << 20)
            break
        except (ftplib.Error, EOFError, OSError) as error:
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise RuntimeError(f"{name}: FTP failed twice ({error})")
            print(f"    connection dropped on {name}; reconnecting")
            ftp = FTP(SINASC_HOST, timeout=300)
            ftp.login()
            ftp.cwd(SINASC_DIR)

    got = partial.stat().st_size
    if got != entry["bytes"]:
        partial.unlink()
        raise RuntimeError(f"{name}: got {got:,} of {entry['bytes']:,} bytes")
    partial.rename(destination)
    limpar_dbc(destination)          # DNSPyyyy.dbc -> DNSPyyyy.dbf.gz without DTNASCMAE
    downloaded += 1
    print(f"  {entry['year']}  {name}  {got / 1e6:.1f} MB")

ftp.quit()
print(f"\n{downloaded} downloaded, {skipped} already here")

# Step 3: look at what arrived — count births per municipality of residence.
# CODMUNRES is the 6-digit DATASUS code, so it needs the same 6->7 lookup that
# SINAN and SIH-SUS use. This is also the check that matters: all 645 São Paulo
# municipalities should show up, because a missing one means no denominator at
# all for it, and a ratio that cannot be computed.
lookup_path = PROJECT_ROOT / "data" / "lookup" / "municipios_sp_estacoes_inmet.xlsx"
if not lookup_path.exists():
    raise RuntimeError(f"Municipality lookup not found at {lookup_path}")
lookup_table = pd.read_excel(lookup_path, engine="calamine")
lookup = {str(int(v))[:-1]: int(v)
          for v in lookup_table["Código Município Completo"].dropna().astype(int)}

newest = arquivos_dbf(source_dir, "DNSP")[-1]
births = Counter()
unmatched = Counter()
with abrir_dbf(newest) as sample_dbf:
    table = dbfread.DBF(str(sample_dbf), encoding="latin1")     # latin1, as always
    print(f"\nSample file: {newest.name}")
    print(f"Rows: {len(table):,}  (one per birth)")
    print(f"Columns ({len(table.field_names)}): {table.field_names}")

    for record in table:
        code_6 = str(record.get("CODMUNRES", "")).strip().zfill(6)
        code_7 = lookup.get(code_6)
        if code_7 is None:
            unmatched[code_6] += 1        # 350000 is "municipality unknown"
            continue
        births[code_7] += 1

sinasc = (pd.DataFrame({"cod_ibge": list(births), "nascidos_vivos": list(births.values())})
          .sort_values("nascidos_vivos", ascending=False)
          .reset_index(drop=True))

print(f"\nMunicipalities with at least one birth: {len(sinasc)} of 645")
print(f"Live births counted: {sinasc['nascidos_vivos'].sum():,}")
print(f"Records with no municipality match: {sum(unmatched.values())} "
      f"{dict(unmatched)}")
print(f"Range: {sinasc['nascidos_vivos'].max():,} in the largest, "
      f"{sinasc['nascidos_vivos'].min()} in the smallest")

sinasc.head(3)